In [48]:
import warnings
warnings.filterwarnings('ignore')

import os
import pickle

import numpy as np
import pandas as pd

from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

from pandas.tseries.offsets import MonthEnd, MonthBegin
from maricovault.MaricoDB import MaricoSnowflake

from joblib import Parallel, delayed

In [49]:
base_dir = '/data/aman_singh/acuuracy_check'
os.chdir(base_dir)

In [50]:
def get_dbconnection(db_name): 

    KEY_VAULT_NAME = "prod-pwd"
    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection

In [51]:
dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.

Credentials retrieved successfully for prod db.


### Helper functions

In [52]:
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment""",
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()

realignment_df = realignment_df[
    realignment_df['channel'].isin(['QCOM', 'QCOM B2C', 'ALL'])]


def realign_pskus(data, column):
    realignment_data = realignment_df.copy()
    assert realignment_data['psku old'].dtype == 'int64'
    assert realignment_data['psku new'].dtype == 'int64'
    realignment_data = realignment_data[['psku old', 'psku new']].drop_duplicates()
    realignment_data = realignment_data.set_index('psku old').to_dict()['psku new']

    
    data[column] = data[column].astype(int)

    for old_psku, new_psku in realignment_data.items():
        data.loc[
            data[column] == old_psku, column
        ] = new_psku

    return data

In [53]:
import glob
import re
import os
import pandas as pd


def read_forecast_file(filepath):
    
    # Read file
    df = pd.read_csv(filepath)
    
    # Extract starting month from filename
    # Example:
    # Marico Ltd._forecast_Sep 2026_to_Dec 2026_sep_blinkit.csv
    match = re.search(
        r'forecast_([A-Za-z]{3})\s+2026_to_',
        os.path.basename(filepath)
    )
    
    if not match:
        raise ValueError(
            f"Could not identify starting month from: {filepath}"
        )
    
    month = match.group(1)
    
    # Convert filename month to month number
    current_month = pd.to_datetime(
        month,
        format='%b'
    ).month
    
    # Next month
    next_month = current_month + 1 if current_month < 12 else 1
    
    # Run month = one month before filename month
    run_month = current_month - 1 if current_month > 1 else 12
    
    # Forecast columns
    current_forecast_col = f'{month} - forecast'
    
    # Find next month's short name
    next_month_name = pd.Timestamp(
        year=2026 if current_month != 12 else 2027,
        month=next_month,
        day=1
    ).strftime('%b')
    
    next_forecast_col = f'{next_month_name} - forecast'
    
    # Check current month column
    if current_forecast_col not in df.columns:
        raise ValueError(
            f"Expected column '{current_forecast_col}' not found in {filepath}. "
            f"Available columns: {df.columns.tolist()}"
        )
    
    # Check next month column
    if next_forecast_col not in df.columns:
        raise ValueError(
            f"Expected column '{next_forecast_col}' not found in {filepath}. "
            f"Available columns: {df.columns.tolist()}"
        )
    
    # Current month data
    df_current = df[
        ['facility_name', 'item_id', current_forecast_col]
    ].copy()
    
    df_current.rename(
        columns={
            'item_id': 'item_code',
            current_forecast_col: 'forecast_quantity'
        },
        inplace=True
    )
    
    df_current['date'] = (
        pd.Timestamp(year=2026, month=current_month, day=1)
        + pd.offsets.MonthEnd(0)
    )
    
    # Next month data
    df_next = df[
        ['facility_name', 'item_id', next_forecast_col]
    ].copy()
    
    df_next.rename(
        columns={
            'item_id': 'item_code',
            next_forecast_col: 'forecast_quantity'
        },
        inplace=True
    )
    
    df_next['date'] = (
        pd.Timestamp(
            year=2026 if current_month != 12 else 2027,
            month=next_month,
            day=1
        )
        + pd.offsets.MonthEnd(0)
    )
    
    # Combine current + next month
    df = pd.concat(
        [df_current, df_next],
        ignore_index=True
    )
    
    # Add chain name
    df['chain_name'] = 'Blinkit'
    
    # Add run month
    df['run_month'] = (
        pd.Timestamp(
            year=2026 if current_month != 1 else 2025,
            month=run_month,
            day=1
        )
        + pd.offsets.MonthEnd(0)
    )
    
    return df


# Path
path = '/data/aman_singh/acuuracy_check/'

# Get all forecast files
files = glob.glob(
    f'{path}Marico Ltd._forecast_*_blinkit.csv'
)

# Read and concatenate
blinkit_forecast = pd.concat(
    [read_forecast_file(file) for file in files],
    ignore_index=True
)

blinkit_forecast

,facility_name,item_code,forecast_quantity,date,chain_name,run_month
0,Super Store Lucknow L4 - Warehouse,10231034,0,2026-05-31,Blinkit,2026-04-30
1,Super Store Lucknow L4 - Warehouse,10044322,2400,2026-05-31,Blinkit,2026-04-30
2,Super Store Lucknow L4 - Warehouse,10196027,61,2026-05-31,Blinkit,2026-04-30
3,Super Store Lucknow L4 - Warehouse,10265690,39,2026-05-31,Blinkit,2026-04-30
4,Super Store Lucknow L4 - Warehouse,10010052,828,2026-05-31,Blinkit,2026-04-30
...,...,...,...,...,...,...
65929,Visakhapatnam V1 - Feeder Warehouse,10180467,0,2026-10-31,Blinkit,2026-08-31
65930,Visakhapatnam V1 - Feeder Warehouse,10203182,349,2026-10-31,Blinkit,2026-08-31
65931,Visakhapatnam V1 - Feeder Warehouse,10000362,867,2026-10-31,Blinkit,2026-08-31
65932,Visakhapatnam V1 - Feeder Warehouse,10004412,1967,2026-10-31,Blinkit,2026-08-31


In [54]:

# import glob
# import re
# import os

# def read_forecast_file(filepath):
    
#     # Read file
#     df = pd.read_csv(filepath)
    
#     # Extract starting month from filename
#     # Example: "..._Sep 2026_to_Dec 2026_..."
#     match = re.search(r'forecast_([A-Za-z]{3})\s+2026_to_', os.path.basename(filepath))
    
#     if not match:
#         raise ValueError(f"Could not identify starting month from: {filepath}")
    
#     month = match.group(1)
    
#     # Forecast column corresponding to starting month
#     forecast_col = f"{month} - forecast"
    
#     if forecast_col not in df.columns:
#         raise ValueError(
#             f"Expected column '{forecast_col}' not found in {filepath}. "
#             f"Available columns: {df.columns.tolist()}"
#         )
    
#     # Keep item_id + required month's forecast
#     df = df[['facility_name','item_id', forecast_col]].copy()
#     df['chain_name'] = 'Blinkit'
    
#     # Rename columns
#     df.rename(
#         columns={
#             'item_id': 'item_code',
#             forecast_col: 'forecast_quantity'
#         },
#         inplace=True
#     )
    
#     # Set date as month-end
#     df['date'] = pd.to_datetime(f'2026-{pd.to_datetime(month, format="%b").month:02d}-01') \
#                  + pd.offsets.MonthEnd(0)
    
#     return df


# # Path
# path = '/data/aman_singh/acuuracy_check/'

# # Get all forecast files
# files = glob.glob(f'{path}Marico Ltd._forecast_*_blinkit.csv')

# # Read and concatenate
# blinkit_forecast = pd.concat(
#     [read_forecast_file(file) for file in files],
#     ignore_index=True
# )

# blinkit_forecast

In [55]:
blinkit_forecast = blinkit_forecast.groupby(['facility_name', 'item_code', 'chain_name', 'run_month','date'], as_index=False)['forecast_quantity'].sum()
blinkit_forecast

,facility_name,item_code,chain_name,run_month,date,forecast_quantity
0,Ahmedabad A2 - Feeder Warehouse,10000059,Blinkit,2026-03-31,2026-04-30,313
1,Ahmedabad A2 - Feeder Warehouse,10000059,Blinkit,2026-03-31,2026-05-31,320
2,Ahmedabad A2 - Feeder Warehouse,10000059,Blinkit,2026-04-30,2026-05-31,341
3,Ahmedabad A2 - Feeder Warehouse,10000059,Blinkit,2026-04-30,2026-06-30,416
4,Ahmedabad A2 - Feeder Warehouse,10000059,Blinkit,2026-05-31,2026-06-30,199
...,...,...,...,...,...,...
65929,Visakhapatnam V1 - Feeder Warehouse,10302659,Blinkit,2026-07-31,2026-09-30,82
65930,Visakhapatnam V1 - Feeder Warehouse,10302659,Blinkit,2026-08-31,2026-09-30,196
65931,Visakhapatnam V1 - Feeder Warehouse,10302659,Blinkit,2026-08-31,2026-10-31,168
65932,Visakhapatnam V1 - Feeder Warehouse,10325055,Blinkit,2026-08-31,2026-09-30,26


In [56]:
#blinkit_unpivoted = pd.concat([blinkit_forecast_apr,blinkit_forecast_may,blinkit_forecast_june])
blinkit_unpivoted = blinkit_forecast.copy()
# blinkit_unpivoted['chain_name'] = 'Blinkit'
# blinkit_unpivoted

In [57]:
def read_swiggy_forecast(filepath):
    
    filename = os.path.basename(filepath)
    
    # Example:
    # MARICO LIMITED_swiggy_sep.xlsx
    month = filename.split('_')[-1].split('.')[0].lower()
    
    # Convert month to month number
    current_month = pd.to_datetime(
        month,
        format='%b'
    ).month
    
    # Next month
    next_month = current_month + 1 if current_month < 12 else 1
    
    # Run month = one month before filename month
    run_month = current_month - 1 if current_month > 1 else 12
    
    # Read file
    df = pd.read_excel(filepath)
    
    # Lowercase column names
    df.columns = df.columns.str.lower()
    
    # Current and next month column names
    current_month_name = pd.Timestamp(
        year=2026,
        month=current_month,
        day=1
    ).strftime('%b').lower()
    
    next_month_name = pd.Timestamp(
        year=2026 if current_month != 12 else 2027,
        month=next_month,
        day=1
    ).strftime('%b').lower()
    
    current_forecast_col = f'{current_month_name}_buy_qty'
    next_forecast_col = f'{next_month_name}_buy_qty'
    
    # Check columns
    if current_forecast_col not in df.columns:
        raise ValueError(
            f"Expected column '{current_forecast_col}' not found in {filepath}. "
            f"Available columns: {df.columns.tolist()}"
        )
    
    if next_forecast_col not in df.columns:
        raise ValueError(
            f"Expected column '{next_forecast_col}' not found in {filepath}. "
            f"Available columns: {df.columns.tolist()}"
        )
    
    # Current month
    df_current = df[
        ['item_code', 'wh_name', current_forecast_col]
    ].copy()
    
    df_current.rename(
        columns={
            'wh_name': 'facility_name',
            current_forecast_col: 'forecast_quantity'
        },
        inplace=True
    )
    
    df_current['date'] = (
        pd.Timestamp(
            year=2026,
            month=current_month,
            day=1
        )
        + pd.offsets.MonthEnd(0)
    )
    
    # Next month
    df_next = df[
        ['item_code', 'wh_name', next_forecast_col]
    ].copy()
    
    df_next.rename(
        columns={
            'wh_name': 'facility_name',
            next_forecast_col: 'forecast_quantity'
        },
        inplace=True
    )
    
    df_next['date'] = (
        pd.Timestamp(
            year=2026 if current_month != 12 else 2027,
            month=next_month,
            day=1
        )
        + pd.offsets.MonthEnd(0)
    )
    
    # Combine current + next month
    df = pd.concat(
        [df_current, df_next],
        ignore_index=True
    )
    
    # Add chain name
    df['chain_name'] = 'Swiggy'
    
    # Add run month
    df['run_month'] = (
        pd.Timestamp(
            year=2026 if current_month != 1 else 2025,
            month=run_month,
            day=1
        )
        + pd.offsets.MonthEnd(0)
    )
    
    return df


files = glob.glob(
    '/data/aman_singh/acuuracy_check/MARICO LIMITED_swiggy_*.xlsx'
)

swiggy_forecast = pd.concat(
    [read_swiggy_forecast(file) for file in files],
    ignore_index=True
)

swiggy_forecast

,item_code,facility_name,forecast_quantity,date,chain_name,run_month
0,17781,BLR DHL,40,2026-07-31,Swiggy,2026-06-30
1,17781,KOC IM1,60,2026-07-31,Swiggy,2026-06-30
2,18103,CBE ECOM,144,2026-07-31,Swiggy,2026-06-30
3,18103,PUN DELHIVERY,48,2026-07-31,Swiggy,2026-06-30
4,18104,HYD IM2,0,2026-07-31,Swiggy,2026-06-30
...,...,...,...,...,...,...
111225,991861,NOI IM1,24,2026-10-31,Swiggy,2026-08-31
111226,999977,BLR IM1,5,2026-10-31,Swiggy,2026-08-31
111227,990631,DLHY GGNFC9,72,2026-10-31,Swiggy,2026-08-31
111228,991861,NAG IM1,24,2026-10-31,Swiggy,2026-08-31


In [58]:
# def read_swiggy_forecast(filepath):
    
#     filename = os.path.basename(filepath)
    
#     # Example:
#     # MARICO LIMITED_swiggy_apr.xlsx
#     month = filename.split('_')[-1].split('.')[0].lower()
    
#     forecast_col = f'{month}_buy_qty'
    
#     df = pd.read_excel(filepath)
#     df.columns = df.columns.str.lower()
    
#     df = df[['item_code','wh_name', forecast_col]].copy()
#     df['chain_name'] = 'Swiggy'
#     df.rename(columns={
#         'wh_name': 'facility_name',
#         forecast_col: 'forecast_quantity'
#     }, inplace=True)
    
#     month_number = pd.to_datetime(month, format='%b').month
    
#     df['date'] = pd.Timestamp(
#         year=2026,
#         month=month_number,
#         day=1
#     ) + pd.offsets.MonthEnd(0)
    
#     return df


# files = glob.glob(
#     '/data/aman_singh/acuuracy_check/MARICO LIMITED_swiggy_*.xlsx'
# )

# swiggy_forecast = pd.concat(
#     [read_swiggy_forecast(file) for file in files],
#     ignore_index=True
# )

# swiggy_forecast

In [59]:
swiggy_forecast['date'].unique()

<DatetimeArray>
['2026-07-31 00:00:00', '2026-08-31 00:00:00', '2026-05-31 00:00:00',
 '2026-06-30 00:00:00', '2026-04-30 00:00:00', '2026-09-30 00:00:00',
 '2026-10-31 00:00:00']
Length: 7, dtype: datetime64[us]

In [60]:
# Swiggy_unpivoted = pd.concat([swiggy_forecast_apr,swiggy_forecast_may,swiggy_forecast_jun])
Swiggy_unpivoted = swiggy_forecast.copy()
# Swiggy_unpivoted['chain_name'] = 'Swiggy'
# Swiggy_unpivoted

In [61]:
blinkit_unpivoted = blinkit_unpivoted.groupby(['chain_name','facility_name','item_code','run_month','date'])['forecast_quantity'].sum().reset_index()
swiggy_unpivoted = Swiggy_unpivoted.groupby(['chain_name','facility_name','item_code','run_month','date'])['forecast_quantity'].sum().reset_index()


In [62]:
swiggy_unpivoted.columns

Index(['chain_name', 'facility_name', 'item_code', 'run_month', 'date',
       'forecast_quantity'],
      dtype='object')

In [63]:
blinkit_unpivoted.columns

Index(['chain_name', 'facility_name', 'item_code', 'run_month', 'date',
       'forecast_quantity'],
      dtype='object')

In [64]:
print(blinkit_unpivoted.dtypes)
print(swiggy_unpivoted.dtypes)

chain_name                   object
facility_name                object
item_code                     int64
run_month            datetime64[us]
date                 datetime64[us]
forecast_quantity             int64
dtype: object
chain_name                   object
facility_name                object
item_code                     int64
run_month            datetime64[us]
date                 datetime64[us]
forecast_quantity             int64
dtype: object


In [65]:
blinkit_unpivoted['date'] = blinkit_unpivoted['date'].astype('datetime64[ns]')
swiggy_unpivoted['date'] = swiggy_unpivoted['date'].astype('datetime64[ns]')

blinkit_unpivoted['run_month'] = blinkit_unpivoted['run_month'].astype('datetime64[ns]')
swiggy_unpivoted['run_month'] = swiggy_unpivoted['run_month'].astype('datetime64[ns]')

chain_forecast_unpivoted = pd.concat([blinkit_unpivoted,swiggy_unpivoted])
# chain_forecast_unpivoted = blinkit_unpivoted.copy()
chain_forecast_unpivoted

,chain_name,facility_name,item_code,run_month,date,forecast_quantity
0,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000059,2026-03-31,2026-04-30,313
1,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000059,2026-03-31,2026-05-31,320
2,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000059,2026-04-30,2026-05-31,341
3,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000059,2026-04-30,2026-06-30,416
4,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000059,2026-05-31,2026-06-30,199
...,...,...,...,...,...,...
111225,Swiggy,VIZ IM1,999977,2026-06-30,2026-08-31,5
111226,Swiggy,VIZ IM1,999977,2026-07-31,2026-08-31,5
111227,Swiggy,VIZ IM1,999977,2026-07-31,2026-09-30,6
111228,Swiggy,VIZ IM1,999977,2026-08-31,2026-09-30,5


In [66]:
mapping = pd.read_excel("/data/aman_singh/mt_forecast/Daily Offtake Tracker - Jul'26.xlsb",sheet_name = 'Mapping')

In [69]:
len_before_merge = len(chain_forecast_unpivoted)
chain_forecast_unpivoted['item_code'] = chain_forecast_unpivoted['item_code'].astype(str)
mapping['asin'] = mapping['asin'].astype(str)
temp = mapping[['platform_name','asin','EAN','PSKU','UOM','Vol per unit']].drop_duplicates()

temp = temp[temp['platform_name'].isin(['Blinkit', 'Swiggy', 'Zepto','ZEPTO'])]
#temp['platform_name'].unique()
duplicates = temp[temp.duplicated(subset="asin", keep=False)]
duplicates



,platform_name,asin,EAN,PSKU,UOM,Vol per unit


In [70]:
temp['PSKU'] = temp['PSKU'].astype(str)
temp['EAN'] = temp['EAN'].astype(str)
temp['UOM'] = temp['UOM'].astype(str)
len_before_merge = len(chain_forecast_unpivoted)
df_chk = chain_forecast_unpivoted.merge(temp,
                  left_on = ['item_code'], right_on = ['asin'], how = 'left')
assert(len_before_merge == len(df_chk))
df_chk['date'] = pd.to_datetime(df_chk['date'])

In [71]:
df_chk

,chain_name,facility_name,item_code,run_month,date,forecast_quantity,platform_name,asin,EAN,PSKU,UOM,Vol per unit
0,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000059,2026-03-31,2026-04-30,313,Blinkit,10000059,8901088000772,718322,KL,5000.0
1,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000059,2026-03-31,2026-05-31,320,Blinkit,10000059,8901088000772,718322,KL,5000.0
2,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000059,2026-04-30,2026-05-31,341,Blinkit,10000059,8901088000772,718322,KL,5000.0
3,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000059,2026-04-30,2026-06-30,416,Blinkit,10000059,8901088000772,718322,KL,5000.0
4,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000059,2026-05-31,2026-06-30,199,Blinkit,10000059,8901088000772,718322,KL,5000.0
...,...,...,...,...,...,...,...,...,...,...,...,...
177159,Swiggy,VIZ IM1,999977,2026-06-30,2026-08-31,5,NaN,NaN,NaN,NaN,NaN,NaN
177160,Swiggy,VIZ IM1,999977,2026-07-31,2026-08-31,5,NaN,NaN,NaN,NaN,NaN,NaN
177161,Swiggy,VIZ IM1,999977,2026-07-31,2026-09-30,6,NaN,NaN,NaN,NaN,NaN,NaN
177162,Swiggy,VIZ IM1,999977,2026-08-31,2026-09-30,5,NaN,NaN,NaN,NaN,NaN,NaN


In [72]:
duplicates = df_chk[df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False)]
duplicates.isnull().sum()

chain_name               0
facility_name            0
item_code                0
run_month                0
date                     0
forecast_quantity        0
platform_name        44822
asin                 44822
EAN                  44822
PSKU                 44822
UOM                  44822
Vol per unit         44822
dtype: int64

In [201]:
# new_mappings = pd.read_excel('/data/aman_singh/acuuracy_check/Alternate_export_file (2).xlsx')
# new_mappings

In [202]:
# nulls = df_chk[(df_chk['PSKU'].isna()) ]#.to_csv('missing_item_to_psku_mappings.csv')#['forecast_quantity'].sum()
# nulls

In [203]:
# nulls[nulls]

In [204]:
# nulls.duplicated(subset=['item_code'], keep=False).sum()

In [205]:
# new_mappings = new_mappings[['Key Account Article Code',
#        'SKU Code']].drop_duplicates()

In [206]:
# new_mappings[new_mappings.duplicated(subset=['Key Account Article Code'], keep=False)].sort_values(by = ['Key Account Article Code'])

In [207]:
# x2 = nulls.merge(new_mappings, left_on = ['item_code'], right_on = ['Key Account Article Code'], how = 'left')

In [208]:
# x2.dtypes

In [209]:
# Swiggy_unpivoted['item_code'] = Swiggy_unpivoted['item_code'].astype(str)
# Swiggy_unpivoted[Swiggy_unpivoted['item_code'].isin(x2['item_code'].unique())].to_csv('missing_item_to_psku_mappingsswiggy.csv', index = False)

In [210]:
# x2.to_csv('missing_item_to_psku_mappings2.csv', index = False)

In [211]:
# x2.to_csv('missing_item_to_psku_mappings2.csv', index = False)

In [36]:
df_chk[(df_chk['chain_name'] == 'Swiggy')]#['forecast_quantity'].sum()

,chain_name,facility_name,item_code,date,forecast_quantity,platform_name,asin,EAN,PSKU,UOM,Vol per unit
32967,Swiggy,AHM DELHIVERY,3,2026-04-30,192,Swiggy,3,89002940,718299,KL,100.0
32968,Swiggy,AHM DELHIVERY,3,2026-05-31,384,Swiggy,3,89002940,718299,KL,100.0
32969,Swiggy,AHM DELHIVERY,3,2026-06-30,0,Swiggy,3,89002940,718299,KL,100.0
32970,Swiggy,AHM DELHIVERY,3,2026-07-31,192,Swiggy,3,89002940,718299,KL,100.0
32971,Swiggy,AHM DELHIVERY,3,2026-08-31,960,Swiggy,3,89002940,718299,KL,100.0
...,...,...,...,...,...,...,...,...,...,...,...
88577,Swiggy,VIZ IM1,999977,2026-05-31,3,NaN,NaN,NaN,NaN,NaN,NaN
88578,Swiggy,VIZ IM1,999977,2026-06-30,2,NaN,NaN,NaN,NaN,NaN,NaN
88579,Swiggy,VIZ IM1,999977,2026-07-31,4,NaN,NaN,NaN,NaN,NaN,NaN
88580,Swiggy,VIZ IM1,999977,2026-08-31,5,NaN,NaN,NaN,NaN,NaN,NaN


In [74]:
#df_chk = df_chk.dropna(subset = ['PSKU'])
df_chk.duplicated(subset=['chain_name','facility_name','PSKU','run_month','date'], keep=False).sum()

46662

In [214]:
# duplicates = df_chk[df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False)]
# duplicates.sort_values(by=['chain_name','facility_name','PSKU','date'])[:60]

In [215]:
# duplicates.sort_values(by=['chain_name','facility_name','PSKU','date']).to_csv('duplicates_swiggy2.csv')

In [216]:
# df_chk = df_chk.sort_values('forecast_quantity', ascending=False) \
#        .drop_duplicates(subset=['chain_name', 'facility_name','PSKU' , 'date'], keep='first')
# df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False).sum()

In [75]:
df_chk.shape

(177164, 12)

In [76]:
df_chk['month_date'] = df_chk['date'] + pd.offsets.MonthEnd(0)

df_chk.rename(columns = {'item_code':'platform_code', 'EAN':'eancode', 'UOM':'uom_reporting',
                         'Vol per unit':'vol_per_unit'},inplace=True)
df_chk['vol_in_lit'] = df_chk['forecast_quantity']*df_chk['vol_per_unit']/1000
df_chk['vol_in_rum'] = df_chk.apply(
    lambda x: x['vol_in_lit'] / 1000 if x['uom_reporting'] in ['KL', 'TO'] else x['vol_in_lit'],
    axis=1
)

df_chk = df_chk.groupby(['chain_name','facility_name', 'PSKU','run_month','month_date'])[['vol_in_rum','forecast_quantity']].sum().reset_index()
df_chk['PSKU'] = df_chk['PSKU'].astype(int)
df_chk.rename(columns = {'PSKU':'parent_material_code'}, inplace = True)
df_chk

,chain_name,facility_name,parent_material_code,run_month,month_date,vol_in_rum,forecast_quantity
0,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-03-31,2026-04-30,9.516,1586
1,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-03-31,2026-05-31,10.206,1701
2,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-04-30,2026-05-31,4.728,788
3,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-04-30,2026-06-30,7.344,1224
4,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-05-31,2026-06-30,5.358,893
...,...,...,...,...,...,...,...
131345,Swiggy,VIZ IM1,811181,2026-08-31,2026-10-31,0.012,12
131346,Swiggy,VIZ IM1,811279,2026-07-31,2026-08-31,0.608,304
131347,Swiggy,VIZ IM1,811279,2026-07-31,2026-09-30,0.000,0
131348,Swiggy,VIZ IM1,811279,2026-08-31,2026-09-30,0.000,0


In [77]:
df_chk.duplicated(subset=['chain_name','facility_name','parent_material_code','run_month','month_date'], keep=False).sum()

0

In [220]:
# facility_to_city_mappings_df = pd.read_excel(
#     r'/data/aman_singh/mt_forecast/City Mappings QCOM.xlsb', 
#     'Sheet1'
# )
# facility_to_city_mappings_df.columns = facility_to_city_mappings_df.columns.str.lower()
# facility_to_city_mappings_df.columns = ['chain', 'facility_name', 'city', 'customer', 'marico_depot',
#        'status', 'depot_name']
# facility_to_city_mappings_df = facility_to_city_mappings_df[
#     facility_to_city_mappings_df['customer'].notna()
# ]
# facility_to_city_mappings_df['facility_name'] = facility_to_city_mappings_df['facility_name'].str.lower()
# facility_to_city_mappings_df['city'] = facility_to_city_mappings_df['city'].str.lower()
# facility_to_city_mappings_df.head()
# customer_depot_mappings_df = pd.read_sql("""
# SELECT DISTINCT customer_code, depot_code, channel_name 
# FROM mst_customer
# WHERE company_code='MIL' AND
#     latest_record_ind=1
# ORDER BY 3, 1, 2
# """,
# prod_conn
# )
# customer_depot_mappings_df.columns = customer_depot_mappings_df.columns.str.lower()
# customer_depot_mappings_df.duplicated(subset=['customer_code']).sum()
# customer_depot_mappings_df['customer_code'] = customer_depot_mappings_df['customer_code'].astype(str)
# facility_to_city_mappings_df['customer'] = facility_to_city_mappings_df['customer'].astype(str)
# customer_depot_mappings_df.dtypes
# facility_to_city_mappings_df.dtypes
# len_before_merge = len(facility_to_city_mappings_df)
# facility_to_city_mappings_df = facility_to_city_mappings_df.merge(
#     customer_depot_mappings_df[['customer_code', 'depot_code']].drop_duplicates().rename(
#         columns={'customer_code': 'customer'}
#     ),
#     on=['customer'],
#     how='left'
# )
# assert len_before_merge == len(facility_to_city_mappings_df)
# del len_before_merge

In [221]:
# facility_to_city_mappings_df#.isnull().sum()

In [222]:
# facility_to_city_mappings_df.rename(columns = {'facility_name':'FC', 'chain':'chain_name'}, inplace = True)
# facility_to_city_mappings_df['FC'] = facility_to_city_mappings_df['FC'].str.lower()

# facility_to_city_mappings_df[facility_to_city_mappings_df.duplicated(subset = ['chain_name','FC'],keep=False)]
# facility_to_city_mappings_df = facility_to_city_mappings_df[['chain_name','FC','depot_code']].drop_duplicates()


In [78]:
facility_to_city_mappings_df = pd.read_excel(
    r'/data/aman_singh/acuuracy_check/City Mappings QCOM.xlsb', 
    'fc_depot_mappings'
)

In [79]:
facility_to_city_mappings_df.rename(columns = {'Facility Name':'FC', 'Marico Depot':'depot_code'}, inplace = True)
facility_to_city_mappings_df['FC'] = facility_to_city_mappings_df['FC'].astype(str)
facility_to_city_mappings_df['FC'] = facility_to_city_mappings_df['FC'].str.replace('\xa0', ' ', regex=True)
facility_to_city_mappings_df['FC'] = facility_to_city_mappings_df['FC'].str.lower()
facility_to_city_mappings_df

,FC,depot_code
0,farukhnagar f2 - feeder warehouse,D115
1,ahmedabad a2 - feeder warehouse,D354
2,hyderabad h3 - feeder warehouse,D530
3,lucknow l5 - feeder warehouse,D113
4,super store hyderabad h2 - warehouse,D530
...,...,...
155,farukhnagar f3 - feeder warehouse,D115
156,hyderabad h4 - feeder warehouse,D530
157,lucknow l6 - feeder warehouse,D113
158,raipur - feeder warehouse,D248


In [80]:
facility_to_city_mappings_df.tail(10)

,FC,depot_code
150,varanasi v2 - feeder warehouse,D113
151,vijayawada - feeder warehouse,D572
152,blr im4 - feeder warehouse,D673
153,ahmedabad a3 - feeder warehouse,D354
154,chennai c6 - feeder warehouse,D674
155,farukhnagar f3 - feeder warehouse,D115
156,hyderabad h4 - feeder warehouse,D530
157,lucknow l6 - feeder warehouse,D113
158,raipur - feeder warehouse,D248
159,blr im4,D673


In [81]:
df_chk.rename(columns = {'facility_name':'FC'},inplace = True)
df_chk['FC'] = df_chk['FC'].str.lower()
df_chk

,chain_name,FC,parent_material_code,run_month,month_date,vol_in_rum,forecast_quantity
0,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-03-31,2026-04-30,9.516,1586
1,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-03-31,2026-05-31,10.206,1701
2,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-04-30,2026-05-31,4.728,788
3,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-04-30,2026-06-30,7.344,1224
4,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-05-31,2026-06-30,5.358,893
...,...,...,...,...,...,...,...
131345,Swiggy,viz im1,811181,2026-08-31,2026-10-31,0.012,12
131346,Swiggy,viz im1,811279,2026-07-31,2026-08-31,0.608,304
131347,Swiggy,viz im1,811279,2026-07-31,2026-09-30,0.000,0
131348,Swiggy,viz im1,811279,2026-08-31,2026-09-30,0.000,0


In [82]:
df_chk[df_chk['chain_name'] == 'Blinkit']['forecast_quantity'].sum()

26273468

In [83]:
chain_forecast_unpivoted[chain_forecast_unpivoted['chain_name'] == 'Blinkit']['forecast_quantity'].sum()

26379618

In [84]:
xy = df_chk.copy()
xy

,chain_name,FC,parent_material_code,run_month,month_date,vol_in_rum,forecast_quantity
0,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-03-31,2026-04-30,9.516,1586
1,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-03-31,2026-05-31,10.206,1701
2,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-04-30,2026-05-31,4.728,788
3,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-04-30,2026-06-30,7.344,1224
4,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-05-31,2026-06-30,5.358,893
...,...,...,...,...,...,...,...
131345,Swiggy,viz im1,811181,2026-08-31,2026-10-31,0.012,12
131346,Swiggy,viz im1,811279,2026-07-31,2026-08-31,0.608,304
131347,Swiggy,viz im1,811279,2026-07-31,2026-09-30,0.000,0
131348,Swiggy,viz im1,811279,2026-08-31,2026-09-30,0.000,0


In [85]:
df_chk = df_chk.merge(facility_to_city_mappings_df, on = ['FC'], how = 'left')
df_chk#.isnull().sum()

,chain_name,FC,parent_material_code,run_month,month_date,vol_in_rum,forecast_quantity,depot_code
0,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-03-31,2026-04-30,9.516,1586,D354
1,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-03-31,2026-05-31,10.206,1701,D354
2,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-04-30,2026-05-31,4.728,788,D354
3,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-04-30,2026-06-30,7.344,1224,D354
4,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-05-31,2026-06-30,5.358,893,D354
...,...,...,...,...,...,...,...,...
131345,Swiggy,viz im1,811181,2026-08-31,2026-10-31,0.012,12,D572
131346,Swiggy,viz im1,811279,2026-07-31,2026-08-31,0.608,304,D572
131347,Swiggy,viz im1,811279,2026-07-31,2026-09-30,0.000,0,D572
131348,Swiggy,viz im1,811279,2026-08-31,2026-09-30,0.000,0,D572


In [86]:
df_chk[df_chk['depot_code'].isna()][['FC','chain_name']].drop_duplicates()#['vol_in_rum'].sum()/df_chk['vol_in_rum'].sum()

,FC,chain_name


In [87]:
df_chk[df_chk['chain_name'] == 'Swiggy']['vol_in_rum'].sum()

361462.848071

In [88]:
df_chk

,chain_name,FC,parent_material_code,run_month,month_date,vol_in_rum,forecast_quantity,depot_code
0,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-03-31,2026-04-30,9.516,1586,D354
1,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-03-31,2026-05-31,10.206,1701,D354
2,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-04-30,2026-05-31,4.728,788,D354
3,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-04-30,2026-06-30,7.344,1224,D354
4,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-05-31,2026-06-30,5.358,893,D354
...,...,...,...,...,...,...,...,...
131345,Swiggy,viz im1,811181,2026-08-31,2026-10-31,0.012,12,D572
131346,Swiggy,viz im1,811279,2026-07-31,2026-08-31,0.608,304,D572
131347,Swiggy,viz im1,811279,2026-07-31,2026-09-30,0.000,0,D572
131348,Swiggy,viz im1,811279,2026-08-31,2026-09-30,0.000,0,D572


In [89]:
material_master_df = pd.read_sql(
    """select * from mst_material 
    where latest_record_ind=1 and company_code='MIL'""",
    prod_conn
)
material_master_df.columns = material_master_df.columns.str.lower()
assert material_master_df.duplicated(
    subset=['company_code', 'material_code']).sum() == 0
material_master_df = material_master_df.rename(columns=
    {'material_group_code': 'brand_code'})
material_master_df['material_code'] = material_master_df['material_code'].astype(np.int64)
material_master_df.duplicated(subset=['material_code', 'parent_material_code', 'brand_code']).sum()

0

In [90]:
# material_master_df[['parent_material_code', 'brand_code']].dtypes
material_master_df['parent_material_code'] = material_master_df['parent_material_code'].astype(int)
material_master_df.loc[material_master_df['parent_material_code'].isin([725930,731857]), 'brand_code'] = 'H&C_ALMND'

# offtake_df.drop(columns = ['brand_code'],inplace = True)
len_before_merge = len(df_chk)
df_chk = df_chk.merge(
    material_master_df[['parent_material_code', 'brand_code']].drop_duplicates(),
    left_on=['parent_material_code'],right_on = ['parent_material_code'],
    how='left'
)
assert len_before_merge == len(df_chk)

In [91]:
df_chk

,chain_name,FC,parent_material_code,run_month,month_date,vol_in_rum,forecast_quantity,depot_code,brand_code
0,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-03-31,2026-04-30,9.516,1586,D354,SAFF GOLD
1,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-03-31,2026-05-31,10.206,1701,D354,SAFF GOLD
2,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-04-30,2026-05-31,4.728,788,D354,SAFF GOLD
3,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-04-30,2026-06-30,7.344,1224,D354,SAFF GOLD
4,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-05-31,2026-06-30,5.358,893,D354,SAFF GOLD
...,...,...,...,...,...,...,...,...,...
131345,Swiggy,viz im1,811181,2026-08-31,2026-10-31,0.012,12,D572,SAF_CDPRS
131346,Swiggy,viz im1,811279,2026-07-31,2026-08-31,0.608,304,D572,SAF_CDPRS
131347,Swiggy,viz im1,811279,2026-07-31,2026-09-30,0.000,0,D572,SAF_CDPRS
131348,Swiggy,viz im1,811279,2026-08-31,2026-09-30,0.000,0,D572,SAF_CDPRS


In [92]:
def read_qtr_ind_rate_table():
    """
    Fetch the club sku information from  DWH_SAP_INDEX_TURNOVER_MONTHWISE table.

    Return:
        qtr_ind_rate_data: pandas dataframe
        - dataframe contains all the results from the index rate table.
    """
    connection = get_dbconnection(db_name='PROD')
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=connection, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    connection.close()
    return qtr_ind_rate



In [93]:
qtr_df = read_qtr_ind_rate_table()
qtr_df.columns = qtr_df.columns.str.lower()
qtr_df.head()


len_before_merge = len(df_chk)

df_chk = df_chk.rename(columns={'material_group_code': 'brand_code'}).merge(
    qtr_df.drop('month_date', axis=1),
    on=['brand_code'],
    how='left'
)

assert len_before_merge == len(df_chk)


Credentials retrieved successfully for prod db.


In [96]:
#df_chk['value'] = df_chk['vol_in_rum']*df_chk['qtr_ind_rate']/10**7
df_chk[df_chk['chain_name'] == 'Blinkit'].groupby(['run_month','month_date'])['value'].sum()

run_month   month_date
2026-03-31  2026-04-30    22.970568
            2026-05-31    26.861270
2026-04-30  2026-05-31    20.848065
            2026-06-30    21.652955
2026-05-31  2026-06-30    20.066041
            2026-07-31    21.539822
2026-06-30  2026-07-31    22.100860
            2026-08-31    25.057049
2026-07-31  2026-08-31    24.093022
            2026-09-30    25.358468
2026-08-31  2026-09-30    23.957488
            2026-10-31    28.627178
Name: value, dtype: float64

In [97]:
df_chk[(df_chk['depot_code'].isna()) & (df_chk['chain_name'] == 'Swiggy')].groupby(['month_date'])['value'].sum()

Series([], Name: value, dtype: float64)

In [98]:
df_chk.groupby(['chain_name'])['value'].sum()

chain_name
Blinkit    283.132786
Swiggy      80.768659
Name: value, dtype: float64

In [99]:
df_chk


,chain_name,FC,parent_material_code,run_month,month_date,vol_in_rum,forecast_quantity,depot_code,brand_code,qtr_ind_rate,value
0,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-03-31,2026-04-30,9.516,1586,D354,SAFF GOLD,138865.260689,0.132144
1,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-03-31,2026-05-31,10.206,1701,D354,SAFF GOLD,138865.260689,0.141726
2,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-04-30,2026-05-31,4.728,788,D354,SAFF GOLD,138865.260689,0.065655
3,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-04-30,2026-06-30,7.344,1224,D354,SAFF GOLD,138865.260689,0.101983
4,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-05-31,2026-06-30,5.358,893,D354,SAFF GOLD,138865.260689,0.074404
...,...,...,...,...,...,...,...,...,...,...,...
131345,Swiggy,viz im1,811181,2026-08-31,2026-10-31,0.012,12,D572,SAF_CDPRS,260000.000000,0.000312
131346,Swiggy,viz im1,811279,2026-07-31,2026-08-31,0.608,304,D572,SAF_CDPRS,260000.000000,0.015808
131347,Swiggy,viz im1,811279,2026-07-31,2026-09-30,0.000,0,D572,SAF_CDPRS,260000.000000,0.000000
131348,Swiggy,viz im1,811279,2026-08-31,2026-09-30,0.000,0,D572,SAF_CDPRS,260000.000000,0.000000


In [104]:
import pandas as pd
import glob
import os

def read_zepto_forecast(filepath):
    
    filename = os.path.basename(filepath)
    
    # Extract month from filename
    # Example: Marico_Limited_projection_sep_zepto.csv
    month_from_file = filename.split('_')[-2].lower()
    
    # Handle full/short month names
    month_from_file = month_from_file[:3]
    
    # Month mentioned in filename
    current_month = pd.to_datetime(
        month_from_file,
        format='%b'
    ).month
    
    # Next month
    next_month = current_month + 1 if current_month < 12 else 1
    
    # Run month = one month before filename month
    run_month = current_month - 1 if current_month > 1 else 12
    
    # Read file
    df = pd.read_csv(filepath)
    
    # Lowercase columns and replace spaces with underscores
    df.columns = (
        df.columns
        .str.lower()
        .str.replace(' ', '')
    )
    
    # Rename forecast column
    if 'final_sku_sales_proj_monthly' in df.columns:
        df.rename(
            columns={'final_sku_sales_proj_monthly': 'forecast_quantity'},
            inplace=True
        )
    
    elif 'projected_qty' in df.columns:
        df.rename(
            columns={'projected_qty': 'forecast_quantity'},
            inplace=True
        )
    
    else:
        raise ValueError(
            f"No forecast column found in {filepath}"
        )
    
    df['forecast_quantity'] = pd.to_numeric(
        df['forecast_quantity']
        .astype(str)
        .str.replace(',', '', regex=False),
        errors='coerce'
    )
    
    # Rename common columns
    df.rename(
        columns={
            'product_variant_id': 'item_code',
            'cluster_dry': 'city'
        },
        inplace=True
    )
    
    # Add chain name
    df['chain_name'] = 'Zepto'
    
    # Normalize month column
    df['month_normalized'] = (
        df['month']
        .astype(str)
        .str.lower()
        .str[:3]
    )
    
    # Convert month to month number
    df['month_number'] = pd.to_datetime(
        df['month_normalized'],
        format='%b',
        errors='coerce'
    ).dt.month
    
    # Keep current month + next month
    df = df[
        df['month_number'].isin([current_month, next_month])
    ].copy()
    
    # Create date
    df['date'] = pd.to_datetime(
        {
            'year': 2026,
            'month': df['month_number'],
            'day': 1
        }
    ) + pd.offsets.MonthEnd(0)
    
    # Add run month
    df['run_month'] = (
        pd.Timestamp(
            year=2026,
            month=run_month,
            day=1
        )
        + pd.offsets.MonthEnd(0)
    )
    
    # Remove helper columns
    df.drop(
        columns=['month_normalized', 'month_number'],
        inplace=True
    )
    
    return df

path = '/data/aman_singh/acuuracy_check/'

files = glob.glob(
    f'{path}Marico_Limited_projection_*_zepto.csv'
)

chain_forecast_zepto = pd.concat(
    [read_zepto_forecast(file) for file in files],
    ignore_index=True
)

chain_forecast_zepto

,month,city,item_code,product_name,category_name,subcategory_name,l3_category_name,brand_name,manufacturer_name,packsize,unit_of_measure,forecast_quantity,chain_name,date,run_month,manufacturer,unit_mrp
0,August,NCR,e011578a-374b-406a-890d-f092005c203d,Saffola Masala Oats |Classic Masala | Anytime ...,Breakfast & Sauces,Muesli & Oats,Oats,Saffola Foods,Marico Limited,38.0,GRAM,85226.0,Zepto,2026-08-31,2026-06-30,NaN,NaN
1,July,NCR,e011578a-374b-406a-890d-f092005c203d,Saffola Masala Oats |Classic Masala | Anytime ...,Breakfast & Sauces,Muesli & Oats,Oats,Saffola Foods,Marico Limited,38.0,GRAM,77254.0,Zepto,2026-07-31,2026-06-30,NaN,NaN
2,August,Hyderabad,709bd327-baf4-4104-a46d-fdba2b80b99c,Parachute 100 % Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,330.0,MILLILITRE,36948.0,Zepto,2026-08-31,2026-06-30,NaN,NaN
3,August,Mumbai,e011578a-374b-406a-890d-f092005c203d,Saffola Masala Oats |Classic Masala | Anytime ...,Breakfast & Sauces,Muesli & Oats,Oats,Saffola Foods,Marico Limited,38.0,GRAM,35949.0,Zepto,2026-08-31,2026-06-30,NaN,NaN
4,August,Mumbai,aaa43e9e-89dc-4c74-acc9-1aa50c57ff1c,Saffola Active Rice Bran & Soyabean Oil | Rich...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Saffola,Marico Limited,850.0,GRAM,33678.0,Zepto,2026-08-31,2026-06-30,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16929,Sept,Bengaluru,19823d3a-a52a-43e9-84b3-498de8bcda2f,Saffola Tasty + Refined Rice bran & Corn Oil |...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,NaN,820.0,GRAM,5266.0,Zepto,2026-09-30,2026-08-31,Marico Limited,158.0
16930,Oct,Hyderabad,19823d3a-a52a-43e9-84b3-498de8bcda2f,Saffola Tasty + Refined Rice bran & Corn Oil |...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,NaN,820.0,GRAM,6259.0,Zepto,2026-10-31,2026-08-31,Marico Limited,158.0
16931,Oct,Coimbatore,19823d3a-a52a-43e9-84b3-498de8bcda2f,Saffola Tasty + Refined Rice bran & Corn Oil |...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,NaN,820.0,GRAM,572.0,Zepto,2026-10-31,2026-08-31,Marico Limited,158.0
16932,Oct,Chennai,19823d3a-a52a-43e9-84b3-498de8bcda2f,Saffola Tasty + Refined Rice bran & Corn Oil |...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,NaN,820.0,GRAM,6501.0,Zepto,2026-10-31,2026-08-31,Marico Limited,158.0


In [105]:
chain_forecast_zepto.groupby(['chain_name','run_month','date'])['forecast_quantity'].sum().reset_index()

,chain_name,run_month,date,forecast_quantity
0,Zepto,2026-03-31,2026-04-30,1282073.0
1,Zepto,2026-03-31,2026-05-31,1363866.0
2,Zepto,2026-04-30,2026-05-31,1280872.0
3,Zepto,2026-04-30,2026-06-30,1430581.0
4,Zepto,2026-05-31,2026-06-30,753604.0
5,Zepto,2026-05-31,2026-07-31,821666.0
6,Zepto,2026-06-30,2026-07-31,1450741.0
7,Zepto,2026-06-30,2026-08-31,1596232.0
8,Zepto,2026-07-31,2026-08-31,1352103.0
9,Zepto,2026-07-31,2026-09-30,1388337.0


In [106]:
chain_forecast_zepto['date'].unique()

<DatetimeArray>
['2026-08-31 00:00:00', '2026-07-31 00:00:00', '2026-05-31 00:00:00',
 '2026-04-30 00:00:00', '2026-06-30 00:00:00', '2026-09-30 00:00:00',
 '2026-10-31 00:00:00']
Length: 7, dtype: datetime64[ns]

In [107]:
# chain_forecast_zepto = pd.concat([chain_forecast_zepto_april,chain_forecast_zepto_may,chain_forecast_zepto_june])
chain_forecast_zepto = chain_forecast_zepto.copy()
chain_forecast_zepto

,month,city,item_code,product_name,category_name,subcategory_name,l3_category_name,brand_name,manufacturer_name,packsize,unit_of_measure,forecast_quantity,chain_name,date,run_month,manufacturer,unit_mrp
0,August,NCR,e011578a-374b-406a-890d-f092005c203d,Saffola Masala Oats |Classic Masala | Anytime ...,Breakfast & Sauces,Muesli & Oats,Oats,Saffola Foods,Marico Limited,38.0,GRAM,85226.0,Zepto,2026-08-31,2026-06-30,NaN,NaN
1,July,NCR,e011578a-374b-406a-890d-f092005c203d,Saffola Masala Oats |Classic Masala | Anytime ...,Breakfast & Sauces,Muesli & Oats,Oats,Saffola Foods,Marico Limited,38.0,GRAM,77254.0,Zepto,2026-07-31,2026-06-30,NaN,NaN
2,August,Hyderabad,709bd327-baf4-4104-a46d-fdba2b80b99c,Parachute 100 % Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,330.0,MILLILITRE,36948.0,Zepto,2026-08-31,2026-06-30,NaN,NaN
3,August,Mumbai,e011578a-374b-406a-890d-f092005c203d,Saffola Masala Oats |Classic Masala | Anytime ...,Breakfast & Sauces,Muesli & Oats,Oats,Saffola Foods,Marico Limited,38.0,GRAM,35949.0,Zepto,2026-08-31,2026-06-30,NaN,NaN
4,August,Mumbai,aaa43e9e-89dc-4c74-acc9-1aa50c57ff1c,Saffola Active Rice Bran & Soyabean Oil | Rich...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Saffola,Marico Limited,850.0,GRAM,33678.0,Zepto,2026-08-31,2026-06-30,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16929,Sept,Bengaluru,19823d3a-a52a-43e9-84b3-498de8bcda2f,Saffola Tasty + Refined Rice bran & Corn Oil |...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,NaN,820.0,GRAM,5266.0,Zepto,2026-09-30,2026-08-31,Marico Limited,158.0
16930,Oct,Hyderabad,19823d3a-a52a-43e9-84b3-498de8bcda2f,Saffola Tasty + Refined Rice bran & Corn Oil |...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,NaN,820.0,GRAM,6259.0,Zepto,2026-10-31,2026-08-31,Marico Limited,158.0
16931,Oct,Coimbatore,19823d3a-a52a-43e9-84b3-498de8bcda2f,Saffola Tasty + Refined Rice bran & Corn Oil |...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,NaN,820.0,GRAM,572.0,Zepto,2026-10-31,2026-08-31,Marico Limited,158.0
16932,Oct,Chennai,19823d3a-a52a-43e9-84b3-498de8bcda2f,Saffola Tasty + Refined Rice bran & Corn Oil |...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,NaN,820.0,GRAM,6501.0,Zepto,2026-10-31,2026-08-31,Marico Limited,158.0


In [108]:
chain_forecast_zepto = chain_forecast_zepto.groupby(['chain_name','city','item_code','run_month','date'])['forecast_quantity'].sum().reset_index()
chain_forecast_zepto

,chain_name,city,item_code,run_month,date,forecast_quantity
0,Zepto,Ahmedabad,00128d5d-d85d-40ed-b70b-c5fa9f10cac0,2026-03-31,2026-04-30,1038.0
1,Zepto,Ahmedabad,00128d5d-d85d-40ed-b70b-c5fa9f10cac0,2026-03-31,2026-05-31,1169.0
2,Zepto,Ahmedabad,00128d5d-d85d-40ed-b70b-c5fa9f10cac0,2026-04-30,2026-05-31,909.0
3,Zepto,Ahmedabad,00128d5d-d85d-40ed-b70b-c5fa9f10cac0,2026-04-30,2026-06-30,1028.0
4,Zepto,Ahmedabad,00128d5d-d85d-40ed-b70b-c5fa9f10cac0,2026-06-30,2026-07-31,424.0
...,...,...,...,...,...,...
16929,Zepto,SAS Nagar,fed7cc7e-7458-4d31-a047-83e88c7c47ee,2026-04-30,2026-06-30,20.0
16930,Zepto,SAS Nagar,fed7cc7e-7458-4d31-a047-83e88c7c47ee,2026-06-30,2026-07-31,28.0
16931,Zepto,SAS Nagar,fed7cc7e-7458-4d31-a047-83e88c7c47ee,2026-06-30,2026-08-31,28.0
16932,Zepto,SAS Nagar,fed7cc7e-7458-4d31-a047-83e88c7c47ee,2026-07-31,2026-08-31,28.0


In [109]:
chain_forecast_zepto['date'] = chain_forecast_zepto['date'].astype('datetime64[ns]')
chain_forecast_zepto['run_month'] = chain_forecast_zepto['run_month'].astype('datetime64[ns]')

In [110]:
zepto_mapping = pd.read_excel('/data/aman_singh/mt_forecast/Q-com Depot-FC-City Mapping v2.0.xlsx', sheet_name = 'Zepto')
zepto_mapping = zepto_mapping[zepto_mapping['Status'] == 'Active']
zepto_mapping = zepto_mapping[['Channel','City', 'Marico Depot']].drop_duplicates()
zepto_mapping.rename(columns = {'Channel':'platform_name', 'City':'city', 'Marico Depot':'depot'}, inplace = True)
zepto_mapping['platform_name'] = zepto_mapping['platform_name'].str.lower()
zepto_mapping['city'] = zepto_mapping['city'].str.lower()
zepto_mapping

,platform_name,city,depot
0,zepto,ahmedabad,D354
2,zepto,indore,D354
3,zepto,mehsana,D354
4,zepto,rajkot,D354
5,zepto,surat,D354
...,...,...,...
110,zepto,pune,D461
111,zepto,bahadurgarh,D115
112,zepto,gurgaon,NaN
113,zepto,raipur,D465


In [111]:
xx = chain_forecast_zepto.copy()

In [112]:
chain_forecast_zepto['city'] = chain_forecast_zepto['city'].str.lower()
chain_forecast_zepto = chain_forecast_zepto.merge(zepto_mapping, on = ['city'], how = 'left')
chain_forecast_zepto

,chain_name,city,item_code,run_month,date,forecast_quantity,platform_name,depot
0,Zepto,ahmedabad,00128d5d-d85d-40ed-b70b-c5fa9f10cac0,2026-03-31,2026-04-30,1038.0,zepto,D354
1,Zepto,ahmedabad,00128d5d-d85d-40ed-b70b-c5fa9f10cac0,2026-03-31,2026-05-31,1169.0,zepto,D354
2,Zepto,ahmedabad,00128d5d-d85d-40ed-b70b-c5fa9f10cac0,2026-04-30,2026-05-31,909.0,zepto,D354
3,Zepto,ahmedabad,00128d5d-d85d-40ed-b70b-c5fa9f10cac0,2026-04-30,2026-06-30,1028.0,zepto,D354
4,Zepto,ahmedabad,00128d5d-d85d-40ed-b70b-c5fa9f10cac0,2026-06-30,2026-07-31,424.0,zepto,D354
...,...,...,...,...,...,...,...,...
16929,Zepto,sas nagar,fed7cc7e-7458-4d31-a047-83e88c7c47ee,2026-04-30,2026-06-30,20.0,zepto,D115
16930,Zepto,sas nagar,fed7cc7e-7458-4d31-a047-83e88c7c47ee,2026-06-30,2026-07-31,28.0,zepto,D115
16931,Zepto,sas nagar,fed7cc7e-7458-4d31-a047-83e88c7c47ee,2026-06-30,2026-08-31,28.0,zepto,D115
16932,Zepto,sas nagar,fed7cc7e-7458-4d31-a047-83e88c7c47ee,2026-07-31,2026-08-31,28.0,zepto,D115


In [113]:
chain_forecast_zepto[chain_forecast_zepto['depot'].isna()]#['city'].unique()

,chain_name,city,item_code,run_month,date,forecast_quantity,platform_name,depot


In [114]:
len_before_merge = len(chain_forecast_zepto)
chain_forecast_zepto['item_code'] = chain_forecast_zepto['item_code'].astype(str)
mapping['asin'] = mapping['asin'].astype(str)
temp = mapping[['platform_name','asin','EAN','PSKU','UOM','Vol per unit']].drop_duplicates()

temp = temp[temp['platform_name'].isin(['Blinkit', 'Swiggy', 'Zepto','ZEPTO'])]
#temp['platform_name'].unique()
duplicates = temp[temp.duplicated(subset="asin", keep=False)]
duplicates



,platform_name,asin,EAN,PSKU,UOM,Vol per unit


In [115]:
temp['PSKU'] = temp['PSKU'].astype(str)
temp['EAN'] = temp['EAN'].astype(str)
temp['UOM'] = temp['UOM'].astype(str)
len_before_merge = len(chain_forecast_zepto)
chain_forecast_zepto = chain_forecast_zepto.merge(temp,
                  left_on = ['item_code'], right_on = ['asin'], how = 'left')
assert(len_before_merge == len(chain_forecast_zepto))
# chain_forecast_zepto['date'] = pd.to_datetime(chain_forecast_zepto['date'])
chain_forecast_zepto

,chain_name,city,item_code,run_month,date,forecast_quantity,platform_name_x,depot,platform_name_y,asin,EAN,PSKU,UOM,Vol per unit
0,Zepto,ahmedabad,00128d5d-d85d-40ed-b70b-c5fa9f10cac0,2026-03-31,2026-04-30,1038.0,zepto,D354,Zepto,00128d5d-d85d-40ed-b70b-c5fa9f10cac0,8901088137034,718287,KL,200.0
1,Zepto,ahmedabad,00128d5d-d85d-40ed-b70b-c5fa9f10cac0,2026-03-31,2026-05-31,1169.0,zepto,D354,Zepto,00128d5d-d85d-40ed-b70b-c5fa9f10cac0,8901088137034,718287,KL,200.0
2,Zepto,ahmedabad,00128d5d-d85d-40ed-b70b-c5fa9f10cac0,2026-04-30,2026-05-31,909.0,zepto,D354,Zepto,00128d5d-d85d-40ed-b70b-c5fa9f10cac0,8901088137034,718287,KL,200.0
3,Zepto,ahmedabad,00128d5d-d85d-40ed-b70b-c5fa9f10cac0,2026-04-30,2026-06-30,1028.0,zepto,D354,Zepto,00128d5d-d85d-40ed-b70b-c5fa9f10cac0,8901088137034,718287,KL,200.0
4,Zepto,ahmedabad,00128d5d-d85d-40ed-b70b-c5fa9f10cac0,2026-06-30,2026-07-31,424.0,zepto,D354,Zepto,00128d5d-d85d-40ed-b70b-c5fa9f10cac0,8901088137034,718287,KL,200.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16929,Zepto,sas nagar,fed7cc7e-7458-4d31-a047-83e88c7c47ee,2026-04-30,2026-06-30,20.0,zepto,D115,NaN,NaN,NaN,NaN,NaN,NaN
16930,Zepto,sas nagar,fed7cc7e-7458-4d31-a047-83e88c7c47ee,2026-06-30,2026-07-31,28.0,zepto,D115,NaN,NaN,NaN,NaN,NaN,NaN
16931,Zepto,sas nagar,fed7cc7e-7458-4d31-a047-83e88c7c47ee,2026-06-30,2026-08-31,28.0,zepto,D115,NaN,NaN,NaN,NaN,NaN,NaN
16932,Zepto,sas nagar,fed7cc7e-7458-4d31-a047-83e88c7c47ee,2026-07-31,2026-08-31,28.0,zepto,D115,NaN,NaN,NaN,NaN,NaN,NaN


In [116]:
xx = chain_forecast_zepto.copy()

In [117]:
chain_forecast_zepto.isnull().sum()

chain_name              0
city                    0
item_code               0
run_month               0
date                    0
forecast_quantity       0
platform_name_x         0
depot                   0
platform_name_y      4534
asin                 4534
EAN                  4534
PSKU                 4534
UOM                  4534
Vol per unit         4534
dtype: int64

In [118]:
chain_forecast_zepto.dropna(subset = ['PSKU'],inplace = True)
chain_forecast_zepto

,chain_name,city,item_code,run_month,date,forecast_quantity,platform_name_x,depot,platform_name_y,asin,EAN,PSKU,UOM,Vol per unit
0,Zepto,ahmedabad,00128d5d-d85d-40ed-b70b-c5fa9f10cac0,2026-03-31,2026-04-30,1038.0,zepto,D354,Zepto,00128d5d-d85d-40ed-b70b-c5fa9f10cac0,8901088137034,718287,KL,200.0
1,Zepto,ahmedabad,00128d5d-d85d-40ed-b70b-c5fa9f10cac0,2026-03-31,2026-05-31,1169.0,zepto,D354,Zepto,00128d5d-d85d-40ed-b70b-c5fa9f10cac0,8901088137034,718287,KL,200.0
2,Zepto,ahmedabad,00128d5d-d85d-40ed-b70b-c5fa9f10cac0,2026-04-30,2026-05-31,909.0,zepto,D354,Zepto,00128d5d-d85d-40ed-b70b-c5fa9f10cac0,8901088137034,718287,KL,200.0
3,Zepto,ahmedabad,00128d5d-d85d-40ed-b70b-c5fa9f10cac0,2026-04-30,2026-06-30,1028.0,zepto,D354,Zepto,00128d5d-d85d-40ed-b70b-c5fa9f10cac0,8901088137034,718287,KL,200.0
4,Zepto,ahmedabad,00128d5d-d85d-40ed-b70b-c5fa9f10cac0,2026-06-30,2026-07-31,424.0,zepto,D354,Zepto,00128d5d-d85d-40ed-b70b-c5fa9f10cac0,8901088137034,718287,KL,200.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16921,Zepto,sas nagar,fc0bdd04-2f61-4a5c-9126-486b4c041a76,2026-07-31,2026-09-30,176.0,zepto,D115,Zepto,fc0bdd04-2f61-4a5c-9126-486b4c041a76,8901088210409,721195,L,250.0
16922,Zepto,sas nagar,fd7408fc-944e-47b5-a6a9-f7e5b8457234,2026-03-31,2026-04-30,120.0,zepto,D115,Zepto,fd7408fc-944e-47b5-a6a9-f7e5b8457234,8901088743495,724723,L,410.0
16923,Zepto,sas nagar,fd7408fc-944e-47b5-a6a9-f7e5b8457234,2026-03-31,2026-05-31,120.0,zepto,D115,Zepto,fd7408fc-944e-47b5-a6a9-f7e5b8457234,8901088743495,724723,L,410.0
16924,Zepto,sas nagar,fd7408fc-944e-47b5-a6a9-f7e5b8457234,2026-04-30,2026-05-31,112.0,zepto,D115,Zepto,fd7408fc-944e-47b5-a6a9-f7e5b8457234,8901088743495,724723,L,410.0


In [ ]:
# chain_forecast_zepto.duplicated(subset=['chain_name','PSKU','date'], keep=False).sum()

2960

In [119]:
chain_forecast_zepto['month_date'] = chain_forecast_zepto['date'] + pd.offsets.MonthEnd(0)

chain_forecast_zepto.rename(columns = {'item_code':'platform_code', 'EAN':'eancode', 'UOM':'uom_reporting',
                         'Vol per unit':'vol_per_unit'},inplace=True)
chain_forecast_zepto['vol_in_lit'] = chain_forecast_zepto['forecast_quantity']*chain_forecast_zepto['vol_per_unit']/1000
chain_forecast_zepto['vol_in_rum'] = chain_forecast_zepto.apply(
    lambda x: x['vol_in_lit'] / 1000 if x['uom_reporting'] in ['KL', 'TO'] else x['vol_in_lit'],
    axis=1
)

chain_forecast_zepto = chain_forecast_zepto.groupby(['chain_name','depot','PSKU','run_month','month_date'])[['vol_in_rum','forecast_quantity']].sum().reset_index()
chain_forecast_zepto['PSKU'] = chain_forecast_zepto['PSKU'].astype(int)
chain_forecast_zepto.rename(columns = {'PSKU':'parent_material_code'}, inplace = True)
chain_forecast_zepto

,chain_name,depot,parent_material_code,run_month,month_date,vol_in_rum,forecast_quantity
0,Zepto,D112,718287,2026-03-31,2026-04-30,2.7076,13538.0
1,Zepto,D112,718287,2026-03-31,2026-05-31,3.0484,15242.0
2,Zepto,D112,718287,2026-04-30,2026-05-31,2.2838,11419.0
3,Zepto,D112,718287,2026-04-30,2026-06-30,2.5826,12913.0
4,Zepto,D112,718287,2026-05-31,2026-06-30,1.0874,5437.0
...,...,...,...,...,...,...,...
11941,Zepto,D674,811181,2026-04-30,2026-06-30,0.1240,124.0
11942,Zepto,D674,811181,2026-06-30,2026-07-31,0.2960,296.0
11943,Zepto,D674,811181,2026-06-30,2026-08-31,0.2960,296.0
11944,Zepto,D674,811181,2026-07-31,2026-08-31,0.3800,380.0


In [120]:
chain_forecast_zepto['forecast_quantity'].sum()

14210968.0

In [121]:
xx['forecast_quantity'].sum()

14464065.0

In [ ]:
# chain_forecast_zepto.to_csv('forecast_zepto.csv')

In [122]:
df_chk

,chain_name,FC,parent_material_code,run_month,month_date,vol_in_rum,forecast_quantity,depot_code,brand_code,qtr_ind_rate,value
0,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-03-31,2026-04-30,9.516,1586,D354,SAFF GOLD,138865.260689,0.132144
1,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-03-31,2026-05-31,10.206,1701,D354,SAFF GOLD,138865.260689,0.141726
2,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-04-30,2026-05-31,4.728,788,D354,SAFF GOLD,138865.260689,0.065655
3,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-04-30,2026-06-30,7.344,1224,D354,SAFF GOLD,138865.260689,0.101983
4,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-05-31,2026-06-30,5.358,893,D354,SAFF GOLD,138865.260689,0.074404
...,...,...,...,...,...,...,...,...,...,...,...
131345,Swiggy,viz im1,811181,2026-08-31,2026-10-31,0.012,12,D572,SAF_CDPRS,260000.000000,0.000312
131346,Swiggy,viz im1,811279,2026-07-31,2026-08-31,0.608,304,D572,SAF_CDPRS,260000.000000,0.015808
131347,Swiggy,viz im1,811279,2026-07-31,2026-09-30,0.000,0,D572,SAF_CDPRS,260000.000000,0.000000
131348,Swiggy,viz im1,811279,2026-08-31,2026-09-30,0.000,0,D572,SAF_CDPRS,260000.000000,0.000000


In [123]:
df_chk['FC'] = df_chk['FC'].str.lower()

In [ ]:
# fc_depot_mapping = {
#     'guwahati g2 - feeder warehouse': 'D236',
#     'indore i2 - feeder warehouse': 'D464',
#     'jaipur j4 - feeder warehouse': 'D314',
#     'mumbai m12 - feeder warehouse': 'D356',
#     'patna p2 - feeder warehouse': 'D233',
#     'pune p3 - feeder warehouse': 'D461',
#     'ranchi r2 - feeder warehouse': 'D234',
#     'surat s2 - feeder warehouse': 'D354',
#     'varanasi v2 - feeder warehouse': 'D113',
#     'vijayawada - feeder warehouse': 'D572',
#     'blr im4 - feeder warehouse': 'D673'
# }
# # Fill only null depot codes
# df_chk.loc[df_chk['depot_code'].isna(), 'depot_code'] = (
#     df_chk.loc[df_chk['depot_code'].isna(), 'FC']
#           .str.lower()
#           .map(fc_depot_mapping)
# )
# df_chk

,chain_name,FC,parent_material_code,month_date,vol_in_rum,forecast_quantity,depot_code,brand_code,qtr_ind_rate,value
0,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-09-30,7.0680,1178,D354,SAFF GOLD,138865.260689,0.098150
1,Blinkit,ahmedabad a2 - feeder warehouse,718312,2026-09-30,0.3420,342,D354,PCNO(R),349274.001420,0.011945
2,Blinkit,ahmedabad a2 - feeder warehouse,718322,2026-09-30,1.1500,230,D354,SAFF KO,168827.536176,0.019415
3,Blinkit,ahmedabad a2 - feeder warehouse,718328,2026-09-30,0.1026,114,D354,SAFF KOCO,123636.889888,0.001269
4,Blinkit,ahmedabad a2 - feeder warehouse,718341,2026-09-30,1.5370,1537,D354,SAFF GOLD,138865.260689,0.021344
...,...,...,...,...,...,...,...,...,...,...
5436,Blinkit,visakhapatnam v1 - feeder warehouse,810520,2026-09-30,0.0010,1,D572,SAF_CDPRS,260000.000000,0.000026
5437,Blinkit,visakhapatnam v1 - feeder warehouse,810673,2026-09-30,0.8400,60,D572,PA_ESS_HO,12860.631072,0.001080
5438,Blinkit,visakhapatnam v1 - feeder warehouse,810674,2026-09-30,0.3220,23,D572,PA_ESS_HO,12860.631072,0.000414
5439,Blinkit,visakhapatnam v1 - feeder warehouse,810971,2026-09-30,0.1500,5,D572,PA_ESS_HO,12860.631072,0.000193


In [124]:
df_chk = df_chk.groupby(['chain_name','depot_code','parent_material_code','run_month','month_date'])[['vol_in_rum','forecast_quantity']].sum().reset_index()
df_chk

,chain_name,depot_code,parent_material_code,run_month,month_date,vol_in_rum,forecast_quantity
0,Blinkit,D112,718288,2026-03-31,2026-04-30,0.018,3
1,Blinkit,D112,718288,2026-03-31,2026-05-31,0.018,3
2,Blinkit,D112,718288,2026-04-30,2026-05-31,0.042,7
3,Blinkit,D112,718288,2026-04-30,2026-06-30,0.042,7
4,Blinkit,D112,718288,2026-05-31,2026-06-30,0.042,7
...,...,...,...,...,...,...,...
75079,Swiggy,D677,811181,2026-08-31,2026-10-31,0.024,24
75080,Swiggy,D677,811279,2026-07-31,2026-08-31,0.312,156
75081,Swiggy,D677,811279,2026-07-31,2026-09-30,0.000,0
75082,Swiggy,D677,811279,2026-08-31,2026-09-30,0.000,0


In [60]:
df_chk.to_csv('blinkit_chain_forecast2.csv')

In [125]:
chain_forecast_zepto

,chain_name,depot,parent_material_code,run_month,month_date,vol_in_rum,forecast_quantity
0,Zepto,D112,718287,2026-03-31,2026-04-30,2.7076,13538.0
1,Zepto,D112,718287,2026-03-31,2026-05-31,3.0484,15242.0
2,Zepto,D112,718287,2026-04-30,2026-05-31,2.2838,11419.0
3,Zepto,D112,718287,2026-04-30,2026-06-30,2.5826,12913.0
4,Zepto,D112,718287,2026-05-31,2026-06-30,1.0874,5437.0
...,...,...,...,...,...,...,...
11941,Zepto,D674,811181,2026-04-30,2026-06-30,0.1240,124.0
11942,Zepto,D674,811181,2026-06-30,2026-07-31,0.2960,296.0
11943,Zepto,D674,811181,2026-06-30,2026-08-31,0.2960,296.0
11944,Zepto,D674,811181,2026-07-31,2026-08-31,0.3800,380.0


In [126]:
df_chk.rename(columns = {'depot_code':'depot'},inplace = True)
final_df = pd.concat([df_chk,chain_forecast_zepto])
final_df

,chain_name,depot,parent_material_code,run_month,month_date,vol_in_rum,forecast_quantity
0,Blinkit,D112,718288,2026-03-31,2026-04-30,0.018,3.0
1,Blinkit,D112,718288,2026-03-31,2026-05-31,0.018,3.0
2,Blinkit,D112,718288,2026-04-30,2026-05-31,0.042,7.0
3,Blinkit,D112,718288,2026-04-30,2026-06-30,0.042,7.0
4,Blinkit,D112,718288,2026-05-31,2026-06-30,0.042,7.0
...,...,...,...,...,...,...,...
11941,Zepto,D674,811181,2026-04-30,2026-06-30,0.124,124.0
11942,Zepto,D674,811181,2026-06-30,2026-07-31,0.296,296.0
11943,Zepto,D674,811181,2026-06-30,2026-08-31,0.296,296.0
11944,Zepto,D674,811181,2026-07-31,2026-08-31,0.380,380.0


In [127]:
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment""",
    dev_conn
)

realignment_df.columns = realignment_df.columns.str.lower()
realignment_df['channel'].unique()
realignment_df = realignment_df[
    realignment_df['channel'].isin(['QCOM', 'QCOM B2C', 'All'])]
def realign_pskus(data, column):
    realignment_data = realignment_df.copy()
    assert realignment_data['psku old'].dtype == 'int64'
    assert realignment_data['psku new'].dtype == 'int64'
    realignment_data = realignment_data[['psku old', 'psku new']].drop_duplicates()
    realignment_data = realignment_data.set_index('psku old').to_dict()['psku new']

    
    data[column] = data[column].astype(int)

    for old_psku, new_psku in realignment_data.items():
        data.loc[
            data[column] == old_psku, column
        ] = new_psku

    return data


In [128]:
final_df.columns

Index(['chain_name', 'depot', 'parent_material_code', 'run_month',
       'month_date', 'vol_in_rum', 'forecast_quantity'],
      dtype='object')

In [ ]:
# base_df
# final_df = base_df.copy()

In [ ]:
final_df['realigned_psku'] = final_df['parent_material_code'].copy()
final_df = realign_pskus(final_df, 'realigned_psku')
base_df = final_df.copy()
final_df = final_df.groupby(
    ['chain_name', 'depot', 'realigned_psku', 'run_month','month_date'],
    as_index=False
)[[ 'vol_in_rum', 'forecast_quantity']].sum()
final_df.duplicated(
    subset=['chain_name', 'depot', 'realigned_psku', 'run_month','month_date']).sum()

0

In [133]:
final_df.isnull().sum()

chain_name           0
depot                0
realigned_psku       0
run_month            0
month_date           0
vol_in_rum           0
forecast_quantity    0
dtype: int64

In [134]:
xx = final_df.copy()

In [ ]:
# final_df.to_csv('qcom_chain_forecast_aug2.csv')

In [135]:
# material_master_df[['parent_material_code', 'brand_code']].dtypes
material_master_df['parent_material_code'] = material_master_df['parent_material_code'].astype(int)
material_master_df.loc[material_master_df['parent_material_code'].isin([725930,731857]), 'brand_code'] = 'H&C_ALMND'

# offtake_df.drop(columns = ['brand_code'],inplace = True)


In [136]:
material_master_df[['parent_material_code', 'brand_code']].drop_duplicates().duplicated(subset=[ 'parent_material_code']).sum()

0

In [137]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    material_master_df[['parent_material_code', 'brand_code']].drop_duplicates(),
    left_on=['realigned_psku'],right_on = ['parent_material_code'],
    how='left'
)
assert len_before_merge == len(final_df)

In [138]:
def read_qtr_ind_rate_table():
    """
    Fetch the club sku information from  DWH_SAP_INDEX_TURNOVER_MONTHWISE table.

    Return:
        qtr_ind_rate_data: pandas dataframe
        - dataframe contains all the results from the index rate table.
    """
    connection = get_dbconnection(db_name='PROD')
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=connection, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    connection.close()
    return qtr_ind_rate



In [139]:
qtr_df = read_qtr_ind_rate_table()
qtr_df.columns = qtr_df.columns.str.lower()
qtr_df.head()


len_before_merge = len(final_df)

final_df = final_df.rename(columns={'material_group_code': 'brand_code'}).merge(
    qtr_df.drop('month_date', axis=1),
    on=['brand_code'],
    how='left'
)

assert len_before_merge == len(final_df)


Credentials retrieved successfully for prod db.


In [141]:
final_df

,chain_name,depot,realigned_psku,run_month,month_date,vol_in_rum,forecast_quantity,parent_material_code,brand_code,qtr_ind_rate
0,Blinkit,D112,718288,2026-03-31,2026-04-30,0.018,3.0,718288,SAFF GOLD,138865.260689
1,Blinkit,D112,718288,2026-03-31,2026-05-31,0.018,3.0,718288,SAFF GOLD,138865.260689
2,Blinkit,D112,718288,2026-04-30,2026-05-31,0.042,7.0,718288,SAFF GOLD,138865.260689
3,Blinkit,D112,718288,2026-04-30,2026-06-30,0.042,7.0,718288,SAFF GOLD,138865.260689
4,Blinkit,D112,718288,2026-05-31,2026-06-30,0.042,7.0,718288,SAFF GOLD,138865.260689
...,...,...,...,...,...,...,...,...,...,...
86569,Zepto,D674,811181,2026-04-30,2026-06-30,0.124,124.0,811181,SAF_CDPRS,260000.000000
86570,Zepto,D674,811181,2026-06-30,2026-07-31,0.296,296.0,811181,SAF_CDPRS,260000.000000
86571,Zepto,D674,811181,2026-06-30,2026-08-31,0.296,296.0,811181,SAF_CDPRS,260000.000000
86572,Zepto,D674,811181,2026-07-31,2026-08-31,0.380,380.0,811181,SAF_CDPRS,260000.000000


In [142]:
final_df['value'] = final_df['vol_in_rum']*final_df['qtr_ind_rate']/10**7
final_df[final_df['chain_name'] == 'Zepto'].groupby(['month_date'])['value'].sum()

month_date
2026-04-30    14.896328
2026-05-31    29.846881
2026-06-30    21.825033
2026-07-31    21.109755
2026-08-31    30.086613
2026-09-30    21.162442
2026-10-31     7.458235
Name: value, dtype: float64

In [143]:
final_df

,chain_name,depot,realigned_psku,run_month,month_date,vol_in_rum,forecast_quantity,parent_material_code,brand_code,qtr_ind_rate,value
0,Blinkit,D112,718288,2026-03-31,2026-04-30,0.018,3.0,718288,SAFF GOLD,138865.260689,0.000250
1,Blinkit,D112,718288,2026-03-31,2026-05-31,0.018,3.0,718288,SAFF GOLD,138865.260689,0.000250
2,Blinkit,D112,718288,2026-04-30,2026-05-31,0.042,7.0,718288,SAFF GOLD,138865.260689,0.000583
3,Blinkit,D112,718288,2026-04-30,2026-06-30,0.042,7.0,718288,SAFF GOLD,138865.260689,0.000583
4,Blinkit,D112,718288,2026-05-31,2026-06-30,0.042,7.0,718288,SAFF GOLD,138865.260689,0.000583
...,...,...,...,...,...,...,...,...,...,...,...
86569,Zepto,D674,811181,2026-04-30,2026-06-30,0.124,124.0,811181,SAF_CDPRS,260000.000000,0.003224
86570,Zepto,D674,811181,2026-06-30,2026-07-31,0.296,296.0,811181,SAF_CDPRS,260000.000000,0.007696
86571,Zepto,D674,811181,2026-06-30,2026-08-31,0.296,296.0,811181,SAF_CDPRS,260000.000000,0.007696
86572,Zepto,D674,811181,2026-07-31,2026-08-31,0.380,380.0,811181,SAF_CDPRS,260000.000000,0.009880


In [144]:
final_df.groupby(['chain_name'])['forecast_quantity'].sum()

chain_name
Blinkit    26273468.0
Swiggy      7741741.0
Zepto      14210968.0
Name: forecast_quantity, dtype: float64

In [145]:
final_df.isnull().sum()

chain_name              0
depot                   0
realigned_psku          0
run_month               0
month_date              0
vol_in_rum              0
forecast_quantity       0
parent_material_code    0
brand_code              0
qtr_ind_rate            0
value                   0
dtype: int64

In [146]:
final_df.to_csv('qcom_chain_forecast_all_months.csv')

### Chain psku primary

In [197]:
qcom_df = pd.read_sql(
    """select * from TRN_MIL_DF_OFT2PRIM_CPSKU """,
    dev_conn
)
qcom_df.columns = qcom_df.columns.str.lower()
qcom_df['run_month'] = pd.to_datetime(qcom_df['run_month'])
qcom_df['month'] = pd.to_datetime(qcom_df['month'])

In [201]:
qcom_df = qcom_df[qcom_df['channel']=='Qcom']

In [203]:
qcom_df['chain'].unique()

array(['Blinkit', 'Swiggy', 'Zepto'], dtype=object)

In [204]:
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment""",
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()

# realignment_df = realignment_df[
#     realignment_df['channel'].isin(['QCOM', 'QCOM B2C', 'ALL'])]


def realign_pskus(data, column, channel='QCOM'):
    realignment_data = realignment_df.copy()
    realignment_data = realignment_data[
        realignment_data['channel'].isin([channel, channel + ' B2C', 'ALL'])
    ]

    assert realignment_data['psku old'].dtype == 'int64'
    assert realignment_data['psku new'].dtype == 'int64'
    realignment_data = realignment_data[['psku old', 'psku new']].drop_duplicates()
    realignment_data = realignment_data.set_index('psku old').to_dict()['psku new']

    
    data[column] = data[column].astype(int)

    for old_psku, new_psku in realignment_data.items():
        data.loc[
            data[column] == old_psku, column
        ] = new_psku

    return data

In [205]:
chain_psku_primary_query = """
SELECT
    CASE
        WHEN MCM.chain = 'Kiranakart Technologies' THEN 'Zepto'
        WHEN MCM.chain = 'Grofers' THEN 'Blinkit'
        ELSE MCM.chain
    END AS chain,
    MM.parent_material_code,
    MM.material_group_code,
    LAST_DAY(MESR.month_date) AS month_date,
    SUM(pri_actuals_vol_rum) AS pri_actuals_vol_rum,
    SUM(pri_apo_plan_vol_rum) AS pri_apo_plan_vol_rum,
    SUM(MESR.sec_apo_plan_vol_rum) AS sec_apo_plan_vol_rum,
    SUM(MESR.sec_actuals_vol_rum) AS sec_actuals_vol_rum
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN 
(
    SELECT
        customer,
        chain_type,
        chain
    FROM
        mst_chain_master
    WHERE
        chain_type = 'E Com B2C' AND 
        chain IN ('Grofers', 'Zepto', 'Kiranakart Technologies', 'Swiggy', 'ZEPTO')
) MCM ON MESR.distributor_code = MCM.customer
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON MESR.material_code = MM.material_code
JOIN
(
    SELECT DISTINCT
        customer_code,
        depot_code
    FROM
        mst_customer
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) CM on MESR.distributor_code = CM.customer_code
where month_date BETWEEN '2025-12-31' AND '2026-07-31' 
GROUP BY 1, 2, 3, 4
ORDER BY 1, 3, 2, 4
"""

depot_psku_primary_df = pd.read_sql(
    chain_psku_primary_query,
    prod_conn
)
depot_psku_primary_df.columns = depot_psku_primary_df.columns.str.lower()
depot_psku_primary_df['parent_material_code'] = depot_psku_primary_df['parent_material_code'].astype(int)
depot_psku_primary_df['month_date'] = pd.to_datetime(depot_psku_primary_df['month_date'])
depot_psku_primary_df = realign_pskus(depot_psku_primary_df.copy(), 'parent_material_code')
depot_psku_primary_df = depot_psku_primary_df.groupby(
    ['chain', 'parent_material_code', 'material_group_code', 'month_date'], as_index=False, dropna=False
).sum()
depot_psku_primary_df = depot_psku_primary_df[depot_psku_primary_df['parent_material_code'] != 715096]
depot_psku_primary_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    if depot_psku_primary_df[col].min() < 0:
        print(col)

for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    depot_psku_primary_df[col] = depot_psku_primary_df[col].clip(lower=0)

pri_actuals_vol_rum
sec_actuals_vol_rum


In [206]:
depot_psku_primary_df[depot_psku_primary_df.duplicated(subset=['chain', 'parent_material_code', 'month_date'], keep=False)]

,chain,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
1263,Blinkit,732296,PABABY_ML,2025-12-31,0.000,0.000000,55.5023,0.000
1264,Blinkit,732296,PABABY_ML,2026-01-31,0.000,161.650358,116.1056,0.000
1265,Blinkit,732296,PABABY_SP,2025-12-31,0.000,0.000000,0.0000,33.600
1266,Blinkit,732296,PABABY_SP,2026-01-31,0.000,0.000000,0.0000,0.000
1273,Blinkit,732297,PABABY_ML,2025-12-31,0.000,0.000000,100.1441,0.000
1274,Blinkit,732297,PABABY_ML,2026-01-31,0.000,397.753905,283.5170,0.000
1275,Blinkit,732297,PABABY_ML,2026-02-28,0.000,0.000000,0.0000,0.000
1276,Blinkit,732297,PABABY_ML,2026-06-30,9.840,0.000000,0.0000,9.840
1277,Blinkit,732297,PABABY_SP,2025-12-31,0.000,0.000000,0.0000,137.760
1278,Blinkit,732297,PABABY_SP,2026-01-31,58.220,0.000000,0.0000,58.220


In [ ]:
# depot_psku_primary_df = depot_psku_primary_df[depot_psku_primary_df['material_group_code']!='PABABY_SP']
# depot_psku_primary_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()


10

In [157]:
depot_psku_primary_df[depot_psku_primary_df.duplicated(subset=['chain', 'parent_material_code', 'month_date'], keep=False)]

,chain,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
1725,Blinkit,811021,PABABY_GM,2026-04-30,52.128,0.000000,0.0000,52.128
1726,Blinkit,811021,PABABY_GM,2026-05-31,0.000,0.000000,0.0000,0.000
1727,Blinkit,811021,PABABY_GM,2026-07-31,69.504,0.000000,0.0000,69.504
1728,Blinkit,811021,PADV_WIPS,2026-04-30,217.200,215.037675,247.6281,0.000
1729,Blinkit,811021,PADV_WIPS,2026-05-31,286.704,421.887556,357.7630,286.704
1731,Blinkit,811021,PADV_WIPS,2026-07-31,0.000,437.094196,544.3660,0.000
2166,Swiggy,718647,LIVON S-R,2025-12-31,0.000,0.000000,277.5746,50.400
2168,Swiggy,718647,LIVON S-R,2026-02-28,122.400,370.404138,296.9990,122.400
2170,Swiggy,718647,LIVON S-R,2026-04-30,0.000,195.462769,195.9974,0.000
2174,Swiggy,718647,LVNPST_ML,2025-12-31,0.000,0.000000,0.0000,0.000


In [207]:
depot_psku_primary_df = depot_psku_primary_df.groupby(['chain', 'parent_material_code', 'month_date'])[['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum',
       'sec_actuals_vol_rum']].sum().reset_index()

In [208]:
depot_psku_primary_df

,chain,parent_material_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,Blinkit,718287,2026-06-30,0.000,1.852098,0.2275,0.000
1,Blinkit,718287,2026-07-31,0.000,2.175176,0.0000,0.000
2,Blinkit,718288,2025-12-31,0.000,0.000000,76.6195,66.186
3,Blinkit,718288,2026-01-31,51.666,49.549496,52.9887,51.666
4,Blinkit,718288,2026-02-28,61.632,55.342992,60.4012,61.632
...,...,...,...,...,...,...,...
5420,Zepto,811279,2026-07-31,0.000,0.321778,0.3156,0.000
5421,Zepto,811287,2026-04-30,26.400,114.301509,0.0000,0.000
5422,Zepto,811287,2026-05-31,60.000,157.918621,122.3085,0.000
5423,Zepto,811287,2026-06-30,21.600,161.147344,179.2152,21.600


In [209]:
depot_psku_primary_df['channel'] = 'Qcom'
depot_psku_primary_df_qcom = depot_psku_primary_df.copy()

In [210]:
depot_psku_primary_df

,chain,parent_material_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum,channel
0,Blinkit,718287,2026-06-30,0.000,1.852098,0.2275,0.000,Qcom
1,Blinkit,718287,2026-07-31,0.000,2.175176,0.0000,0.000,Qcom
2,Blinkit,718288,2025-12-31,0.000,0.000000,76.6195,66.186,Qcom
3,Blinkit,718288,2026-01-31,51.666,49.549496,52.9887,51.666,Qcom
4,Blinkit,718288,2026-02-28,61.632,55.342992,60.4012,61.632,Qcom
...,...,...,...,...,...,...,...,...
5420,Zepto,811279,2026-07-31,0.000,0.321778,0.3156,0.000,Qcom
5421,Zepto,811287,2026-04-30,26.400,114.301509,0.0000,0.000,Qcom
5422,Zepto,811287,2026-05-31,60.000,157.918621,122.3085,0.000,Qcom
5423,Zepto,811287,2026-06-30,21.600,161.147344,179.2152,21.600,Qcom


In [211]:
actuals_df = depot_psku_primary_df.copy()

In [212]:
actuals_df.duplicated(subset=['channel','chain', 'parent_material_code','month_date']).sum()

0

In [213]:
actuals_df = actuals_df.groupby(['channel','chain', 'parent_material_code',
        'month_date'])[['pri_actuals_vol_rum',
       'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']].sum().reset_index()
actuals_df

,channel,chain,parent_material_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,Qcom,Blinkit,718287,2026-06-30,0.000,1.852098,0.2275,0.000
1,Qcom,Blinkit,718287,2026-07-31,0.000,2.175176,0.0000,0.000
2,Qcom,Blinkit,718288,2025-12-31,0.000,0.000000,76.6195,66.186
3,Qcom,Blinkit,718288,2026-01-31,51.666,49.549496,52.9887,51.666
4,Qcom,Blinkit,718288,2026-02-28,61.632,55.342992,60.4012,61.632
...,...,...,...,...,...,...,...,...
5420,Qcom,Zepto,811279,2026-07-31,0.000,0.321778,0.3156,0.000
5421,Qcom,Zepto,811287,2026-04-30,26.400,114.301509,0.0000,0.000
5422,Qcom,Zepto,811287,2026-05-31,60.000,157.918621,122.3085,0.000
5423,Qcom,Zepto,811287,2026-06-30,21.600,161.147344,179.2152,21.600


In [214]:
actuals_df.columns

Index(['channel', 'chain', 'parent_material_code', 'month_date',
       'pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum',
       'sec_actuals_vol_rum'],
      dtype='object')

In [215]:
actuals_df = actuals_df.rename(columns={
    'parent_material_code': 'psku',
    'sec_apo_plan_vol_rum': 'Consensus Vol',
    'sec_actuals_vol_rum': 'Actuals Vol',
    'month_date': 'month'
})

In [216]:
actuals_df.head()

,channel,chain,psku,month,pri_actuals_vol_rum,pri_apo_plan_vol_rum,Consensus Vol,Actuals Vol
0,Qcom,Blinkit,718287,2026-06-30,0.000,1.852098,0.2275,0.000
1,Qcom,Blinkit,718287,2026-07-31,0.000,2.175176,0.0000,0.000
2,Qcom,Blinkit,718288,2025-12-31,0.000,0.000000,76.6195,66.186
3,Qcom,Blinkit,718288,2026-01-31,51.666,49.549496,52.9887,51.666
4,Qcom,Blinkit,718288,2026-02-28,61.632,55.342992,60.4012,61.632


In [ ]:
#qcom_df.rename(columns = {'Channel':'channel'}, inplace = True)

In [217]:
qcom_df['psku'] = qcom_df['psku'].astype(int)
actuals_df['psku'] = actuals_df['psku'].astype(int)

In [218]:
actuals_df['channel'] = actuals_df['channel'].str.upper()
qcom_df['channel'] = qcom_df['channel'].str.upper()
actuals_df['chain'] = actuals_df['chain'].str.lower()
qcom_df['chain'] = qcom_df['chain'].str.lower()


In [ ]:
# qcom_df['channel'].unique()

array(['QCOM', 'ECOM'], dtype=object)

In [ ]:
# actuals_df = actuals_df[actuals_df['month_date'] == '2025-12-31']
# actuals_df

In [219]:
x = qcom_df.copy()

In [220]:
len_before_merge = len(qcom_df)
qcom_df = qcom_df.merge(
    actuals_df.drop([  'pri_actuals_vol_rum', 'pri_apo_plan_vol_rum'], axis=1),
    on=['channel','chain', 'psku', 'month'],
    how='left'
)
assert len_before_merge == len(qcom_df)
del len_before_merge

In [221]:
def read_qtr_ind_rate_table():
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=prod_conn, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    
    return qtr_ind_rate

qtr_ind_rate_df = read_qtr_ind_rate_table()
qtr_ind_rate_df.head()

,month_date,brand_code,qtr_ind_rate
0,2027-03-31,CMX_WELPD,3014.000
1,2027-03-31,4700_BCPC,222.000
2,2027-03-31,CMX_PRTPD,3014.000
3,2027-03-31,PA_CN_HGO,488.152
4,2027-03-31,TRU_RAWDF,800.000


In [222]:
len_before_merge = len(qcom_df)
qcom_df = qcom_df.merge(
    qtr_ind_rate_df.drop('month_date', axis=1).rename(
        columns={'brand_code': 'brand', 'qtr_ind_rate': 'Index Rate'}
    ),
    on=['brand'], 
    how='left'
)
assert len_before_merge == len(qcom_df)
del len_before_merge

In [223]:
qcom_df.isna().sum()

month                         0
chain                         0
psku                          0
brand                         0
portfolio                     0
m month                       0
run_month                     0
calculated primary vol        0
channel                       0
Consensus Vol             26989
Actuals Vol               26989
Index Rate                    0
dtype: int64

In [224]:
qcom_df['Consensus Vol'] = qcom_df['Consensus Vol'].fillna(0)
qcom_df['Actuals Vol'] = qcom_df['Actuals Vol'].fillna(0)

In [225]:
qcom_df['Consensus Vol'].max()

22636.4666

In [226]:
qcom_df.rename(columns={'calculated primary vol': 'Stat Vol'}, inplace=True)
qcom_df

,month,chain,psku,brand,portfolio,m month,run_month,Stat Vol,channel,Consensus Vol,Actuals Vol,Index Rate
0,2025-12-31,blinkit,718287,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,0.000,349274.001420
1,2025-12-31,blinkit,718288,SAFF GOLD,Saffola Oils,M,2025-12-31,76.619500,QCOM,76.6195,66.186,138865.260689
2,2025-12-31,blinkit,718297,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,0.000,349274.001420
3,2025-12-31,blinkit,718299,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,0.000,349274.001420
4,2025-12-31,blinkit,718300,PCNO FLEX,CNO,M,2025-12-31,0.000000,QCOM,0.0000,0.000,230000.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
39469,2026-09-30,zepto,811169,SW_SGPRF,Male Grooming,M+3,2026-06-30,0.000000,QCOM,0.0000,0.000,1712.605337
39470,2026-09-30,zepto,811181,SAF_CDPRS,Saffola Oils,M+3,2026-06-30,0.184668,QCOM,0.0000,0.000,260000.000000
39471,2026-09-30,zepto,811267,PA_ESS_HO,Hair Oils,M+3,2026-06-30,0.000000,QCOM,0.0000,0.000,12860.631072
39472,2026-09-30,zepto,811269,PA_ESS_HO,Hair Oils,M+3,2026-06-30,0.000000,QCOM,0.0000,0.000,12860.631072


In [227]:
qcom_df['Stat Val'] = qcom_df['Stat Vol'] * qcom_df['Index Rate'] / (10 ** 7)
qcom_df['Consensus Val'] = qcom_df['Consensus Vol'] * qcom_df['Index Rate'] / (10 ** 7)
qcom_df['Actuals Val'] = qcom_df['Actuals Vol'] * qcom_df['Index Rate'] / (10 ** 7)

In [228]:
qcom_df

,month,chain,psku,brand,portfolio,m month,run_month,Stat Vol,channel,Consensus Vol,Actuals Vol,Index Rate,Stat Val,Consensus Val,Actuals Val
0,2025-12-31,blinkit,718287,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,0.000,349274.001420,0.000000,0.000000,0.000000
1,2025-12-31,blinkit,718288,SAFF GOLD,Saffola Oils,M,2025-12-31,76.619500,QCOM,76.6195,66.186,138865.260689,1.063979,1.063979,0.919094
2,2025-12-31,blinkit,718297,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,0.000,349274.001420,0.000000,0.000000,0.000000
3,2025-12-31,blinkit,718299,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,0.000,349274.001420,0.000000,0.000000,0.000000
4,2025-12-31,blinkit,718300,PCNO FLEX,CNO,M,2025-12-31,0.000000,QCOM,0.0000,0.000,230000.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39469,2026-09-30,zepto,811169,SW_SGPRF,Male Grooming,M+3,2026-06-30,0.000000,QCOM,0.0000,0.000,1712.605337,0.000000,0.000000,0.000000
39470,2026-09-30,zepto,811181,SAF_CDPRS,Saffola Oils,M+3,2026-06-30,0.184668,QCOM,0.0000,0.000,260000.000000,0.004801,0.000000,0.000000
39471,2026-09-30,zepto,811267,PA_ESS_HO,Hair Oils,M+3,2026-06-30,0.000000,QCOM,0.0000,0.000,12860.631072,0.000000,0.000000,0.000000
39472,2026-09-30,zepto,811269,PA_ESS_HO,Hair Oils,M+3,2026-06-30,0.000000,QCOM,0.0000,0.000,12860.631072,0.000000,0.000000,0.000000


In [229]:
qcom_df[(qcom_df['month'] == '2026-07-31') & (qcom_df['channel']=='QCOM') & (qcom_df['run_month']=='2026-06-30')]['Actuals Val'].sum()

33.574161831200286

In [230]:
qcom_df['Stat Error'] = qcom_df['Stat Val'] - qcom_df['Actuals Val']
qcom_df['Consensus Error'] = qcom_df['Consensus Val'] - qcom_df['Actuals Val']

qcom_df['Stat Abs Error'] = np.abs(qcom_df['Stat Error'])
qcom_df['Consensus Abs Error'] = np.abs(qcom_df['Consensus Error'])

In [231]:
qcom_df

,month,chain,psku,brand,portfolio,m month,run_month,Stat Vol,channel,Consensus Vol,Actuals Vol,Index Rate,Stat Val,Consensus Val,Actuals Val,Stat Error,Consensus Error,Stat Abs Error,Consensus Abs Error
0,2025-12-31,blinkit,718287,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,0.000,349274.001420,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2025-12-31,blinkit,718288,SAFF GOLD,Saffola Oils,M,2025-12-31,76.619500,QCOM,76.6195,66.186,138865.260689,1.063979,1.063979,0.919094,0.144885,0.144885,0.144885,0.144885
2,2025-12-31,blinkit,718297,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,0.000,349274.001420,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,2025-12-31,blinkit,718299,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,0.000,349274.001420,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,2025-12-31,blinkit,718300,PCNO FLEX,CNO,M,2025-12-31,0.000000,QCOM,0.0000,0.000,230000.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39469,2026-09-30,zepto,811169,SW_SGPRF,Male Grooming,M+3,2026-06-30,0.000000,QCOM,0.0000,0.000,1712.605337,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
39470,2026-09-30,zepto,811181,SAF_CDPRS,Saffola Oils,M+3,2026-06-30,0.184668,QCOM,0.0000,0.000,260000.000000,0.004801,0.000000,0.000000,0.004801,0.000000,0.004801,0.000000
39471,2026-09-30,zepto,811267,PA_ESS_HO,Hair Oils,M+3,2026-06-30,0.000000,QCOM,0.0000,0.000,12860.631072,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
39472,2026-09-30,zepto,811269,PA_ESS_HO,Hair Oils,M+3,2026-06-30,0.000000,QCOM,0.0000,0.000,12860.631072,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [232]:
qcom_df = qcom_df[qcom_df['m month'] == 'M+1']
qcom_df

,month,chain,psku,brand,portfolio,m month,run_month,Stat Vol,channel,Consensus Vol,Actuals Vol,Index Rate,Stat Val,Consensus Val,Actuals Val,Stat Error,Consensus Error,Stat Abs Error,Consensus Abs Error
1571,2026-01-31,blinkit,718287,PCNO(R),CNO,M+1,2025-12-31,0.000000,QCOM,0.0000,0.000,349274.001420,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1572,2026-01-31,blinkit,718288,SAFF GOLD,Saffola Oils,M+1,2025-12-31,52.041436,QCOM,52.9887,51.666,138865.260689,0.722675,0.735829,0.717461,0.005213,0.018368,0.005213,0.018368
1573,2026-01-31,blinkit,718297,PCNO(R),CNO,M+1,2025-12-31,0.000000,QCOM,0.0000,0.000,349274.001420,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1574,2026-01-31,blinkit,718299,PCNO(R),CNO,M+1,2025-12-31,0.000000,QCOM,0.0000,0.000,349274.001420,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1575,2026-01-31,blinkit,718300,PCNO FLEX,CNO,M+1,2025-12-31,0.000000,QCOM,0.0000,0.000,230000.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36163,2026-07-31,zepto,811169,SW_SGPRF,Male Grooming,M+1,2026-06-30,0.000000,QCOM,0.0000,0.000,1712.605337,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
36164,2026-07-31,zepto,811181,SAF_CDPRS,Saffola Oils,M+1,2026-06-30,0.185667,QCOM,0.3045,1.020,260000.000000,0.004827,0.007917,0.026520,-0.021693,-0.018603,0.021693,0.018603
36165,2026-07-31,zepto,811267,PA_ESS_HO,Hair Oils,M+1,2026-06-30,0.000000,QCOM,0.0000,0.000,12860.631072,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
36166,2026-07-31,zepto,811269,PA_ESS_HO,Hair Oils,M+1,2026-06-30,0.000000,QCOM,0.0000,0.000,12860.631072,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [233]:
xx = qcom_df.copy()

In [234]:
qcom_df = qcom_df[qcom_df['run_month'].isin(['2026-03-31', '2026-04-30', '2026-05-31', '2026-06-30'])]
qcom_df

,month,chain,psku,brand,portfolio,m month,run_month,Stat Vol,channel,Consensus Vol,Actuals Vol,Index Rate,Stat Val,Consensus Val,Actuals Val,Stat Error,Consensus Error,Stat Abs Error,Consensus Abs Error
14975,2026-04-30,blinkit,718287,PCNO(R),CNO,M+1,2026-03-31,0.000000,QCOM,0.0000,0.000,349274.001420,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
14976,2026-04-30,blinkit,718288,SAFF GOLD,Saffola Oils,M+1,2026-03-31,62.778362,QCOM,60.9032,91.068,138865.260689,0.871773,0.845734,1.264618,-0.392845,-0.418884,0.392845,0.418884
14977,2026-04-30,blinkit,718297,PCNO(R),CNO,M+1,2026-03-31,0.000000,QCOM,0.0000,0.000,349274.001420,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
14978,2026-04-30,blinkit,718299,PCNO(R),CNO,M+1,2026-03-31,0.000000,QCOM,0.0000,0.000,349274.001420,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
14979,2026-04-30,blinkit,718300,PCNO FLEX,CNO,M+1,2026-03-31,0.000000,QCOM,0.0000,0.000,230000.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36163,2026-07-31,zepto,811169,SW_SGPRF,Male Grooming,M+1,2026-06-30,0.000000,QCOM,0.0000,0.000,1712.605337,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
36164,2026-07-31,zepto,811181,SAF_CDPRS,Saffola Oils,M+1,2026-06-30,0.185667,QCOM,0.3045,1.020,260000.000000,0.004827,0.007917,0.026520,-0.021693,-0.018603,0.021693,0.018603
36165,2026-07-31,zepto,811267,PA_ESS_HO,Hair Oils,M+1,2026-06-30,0.000000,QCOM,0.0000,0.000,12860.631072,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
36166,2026-07-31,zepto,811269,PA_ESS_HO,Hair Oils,M+1,2026-06-30,0.000000,QCOM,0.0000,0.000,12860.631072,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [190]:
final_df = final_df.groupby(['chain_name', 'realigned_psku', 'run_month','month_date'], as_index=False)[['vol_in_rum', 'forecast_quantity']].sum()
final_df.rename(columns={'chain_name':'chain','realigned_psku': 'psku','month_date':'month'}, inplace=True)
final_df

,chain,psku,run_month,month,vol_in_rum,forecast_quantity
0,Blinkit,718288,2026-03-31,2026-04-30,87.834,14639.0
1,Blinkit,718288,2026-03-31,2026-05-31,102.960,17160.0
2,Blinkit,718288,2026-04-30,2026-05-31,103.344,17224.0
3,Blinkit,718288,2026-04-30,2026-06-30,93.480,15580.0
4,Blinkit,718288,2026-05-31,2026-06-30,64.560,10760.0
...,...,...,...,...,...,...
6727,Zepto,811181,2026-07-31,2026-09-30,1.600,1600.0
6728,Zepto,811287,2026-06-30,2026-07-31,16.800,168.0
6729,Zepto,811287,2026-06-30,2026-08-31,16.800,168.0
6730,Zepto,811287,2026-07-31,2026-08-31,28.800,288.0


In [238]:
final_df = final_df[final_df['run_month'].isin(['2026-03-31', '2026-04-30', '2026-05-31', '2026-06-30'])]
final_df

,chain,psku,run_month,month,vol_in_rum,forecast_quantity
0,blinkit,718288,2026-03-31,2026-04-30,87.834,14639.0
1,blinkit,718288,2026-03-31,2026-05-31,102.960,17160.0
2,blinkit,718288,2026-04-30,2026-05-31,103.344,17224.0
3,blinkit,718288,2026-04-30,2026-06-30,93.480,15580.0
4,blinkit,718288,2026-05-31,2026-06-30,64.560,10760.0
...,...,...,...,...,...,...
6723,zepto,811181,2026-04-30,2026-06-30,0.572,572.0
6724,zepto,811181,2026-06-30,2026-07-31,1.252,1252.0
6725,zepto,811181,2026-06-30,2026-08-31,1.252,1252.0
6728,zepto,811287,2026-06-30,2026-07-31,16.800,168.0


In [240]:
mappings = {}

for run_month in final_df['run_month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 9):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


mappings   
final_df['m_month_chain'] = final_df.apply(
    lambda x: mappings[x['run_month']].get(
        x['month']
    ), axis=1
)

In [243]:
final_df['chain'] = final_df['chain'].str.lower()
merged_df = qcom_df.merge(final_df, on=['chain', 'psku', 'run_month'], how='left')
merged_df

,month_x,chain,psku,brand,portfolio,m month,run_month,Stat Vol,channel,Consensus Vol,...,Consensus Val,Actuals Val,Stat Error,Consensus Error,Stat Abs Error,Consensus Abs Error,month_y,vol_in_rum,forecast_quantity,m_month_chain
0,2026-04-30,blinkit,718287,PCNO(R),CNO,M+1,2026-03-31,0.000000,QCOM,0.0000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaT,NaN,NaN,NaN
1,2026-04-30,blinkit,718288,SAFF GOLD,Saffola Oils,M+1,2026-03-31,62.778362,QCOM,60.9032,...,0.845734,1.264618,-0.392845,-0.418884,0.392845,0.418884,2026-04-30,87.834,14639.0,M+1
2,2026-04-30,blinkit,718288,SAFF GOLD,Saffola Oils,M+1,2026-03-31,62.778362,QCOM,60.9032,...,0.845734,1.264618,-0.392845,-0.418884,0.392845,0.418884,2026-05-31,102.960,17160.0,M+2
3,2026-04-30,blinkit,718297,PCNO(R),CNO,M+1,2026-03-31,0.000000,QCOM,0.0000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaT,NaN,NaN,NaN
4,2026-04-30,blinkit,718299,PCNO(R),CNO,M+1,2026-03-31,0.000000,QCOM,0.0000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaT,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8665,2026-07-31,zepto,811181,SAF_CDPRS,Saffola Oils,M+1,2026-06-30,0.185667,QCOM,0.3045,...,0.007917,0.026520,-0.021693,-0.018603,0.021693,0.018603,2026-07-31,1.252,1252.0,M+1
8666,2026-07-31,zepto,811181,SAF_CDPRS,Saffola Oils,M+1,2026-06-30,0.185667,QCOM,0.3045,...,0.007917,0.026520,-0.021693,-0.018603,0.021693,0.018603,2026-08-31,1.252,1252.0,M+2
8667,2026-07-31,zepto,811267,PA_ESS_HO,Hair Oils,M+1,2026-06-30,0.000000,QCOM,0.0000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaT,NaN,NaN,NaN
8668,2026-07-31,zepto,811269,PA_ESS_HO,Hair Oils,M+1,2026-06-30,0.000000,QCOM,0.0000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaT,NaN,NaN,NaN


In [244]:
merged_df['Chain Val'] = merged_df['vol_in_rum'] * merged_df['Index Rate'] / (10 ** 7)
merged_df['Chain Error'] = merged_df['Chain Val'] - merged_df['Actuals Val']

merged_df['Chain Abs Error'] = np.abs(merged_df['Chain Error'])


In [245]:
merged_df

,month_x,chain,psku,brand,portfolio,m month,run_month,Stat Vol,channel,Consensus Vol,...,Consensus Error,Stat Abs Error,Consensus Abs Error,month_y,vol_in_rum,forecast_quantity,m_month_chain,Chain Val,Chain Error,Chain Abs Error
0,2026-04-30,blinkit,718287,PCNO(R),CNO,M+1,2026-03-31,0.000000,QCOM,0.0000,...,0.000000,0.000000,0.000000,NaT,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-04-30,blinkit,718288,SAFF GOLD,Saffola Oils,M+1,2026-03-31,62.778362,QCOM,60.9032,...,-0.418884,0.392845,0.418884,2026-04-30,87.834,14639.0,M+1,1.219709,-0.044909,0.044909
2,2026-04-30,blinkit,718288,SAFF GOLD,Saffola Oils,M+1,2026-03-31,62.778362,QCOM,60.9032,...,-0.418884,0.392845,0.418884,2026-05-31,102.960,17160.0,M+2,1.429757,0.165139,0.165139
3,2026-04-30,blinkit,718297,PCNO(R),CNO,M+1,2026-03-31,0.000000,QCOM,0.0000,...,0.000000,0.000000,0.000000,NaT,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-04-30,blinkit,718299,PCNO(R),CNO,M+1,2026-03-31,0.000000,QCOM,0.0000,...,0.000000,0.000000,0.000000,NaT,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8665,2026-07-31,zepto,811181,SAF_CDPRS,Saffola Oils,M+1,2026-06-30,0.185667,QCOM,0.3045,...,-0.018603,0.021693,0.018603,2026-07-31,1.252,1252.0,M+1,0.032552,0.006032,0.006032
8666,2026-07-31,zepto,811181,SAF_CDPRS,Saffola Oils,M+1,2026-06-30,0.185667,QCOM,0.3045,...,-0.018603,0.021693,0.018603,2026-08-31,1.252,1252.0,M+2,0.032552,0.006032,0.006032
8667,2026-07-31,zepto,811267,PA_ESS_HO,Hair Oils,M+1,2026-06-30,0.000000,QCOM,0.0000,...,0.000000,0.000000,0.000000,NaT,NaN,NaN,NaN,NaN,NaN,NaN
8668,2026-07-31,zepto,811269,PA_ESS_HO,Hair Oils,M+1,2026-06-30,0.000000,QCOM,0.0000,...,0.000000,0.000000,0.000000,NaT,NaN,NaN,NaN,NaN,NaN,NaN


In [248]:
merged_df[(merged_df['month_x'] == '2026-07-31') & (merged_df['channel']=='QCOM') & (merged_df['run_month']=='2026-06-30')]['Chain Val'].sum()


88.71760118785923

In [250]:
merged_df.to_excel('qcom_stat_chain_forecast_acc_comparison.xlsx', index=False)

In [ ]:
qcom_df['stat_bias'] = qcom_df['Stat Error']/qcom_df['Actuals Val']
qcom_df = qcom_df.fillna(0)

import numpy as np
import pandas as pd

qcom_df['stat_bias'] = (
    qcom_df['stat_bias']
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

bins = [-np.inf, -0.15, -0.10, -0.05, 0, 0.05, 0.10, 0.15, np.inf]
labels = [
    '< -15%',
    '-15% to -10%',
    '-10% to -5%',
    '-5% to 0%',
    '0% to 5%',
    '5% to 10%',
    '10% to 15%',
    '> 15%'
]

qcom_df['stat_bias_bucket'] = pd.cut(
    qcom_df['stat_bias'],
    bins=bins,
    labels=labels,
    right=False  
)


### The end

In [71]:
primary = pd.read_csv('/data/aman_singh/acuuracy_check/QCOM Chain PSKU OTP Output/live_runs/QCOM Chain Depot PSKU Primary_live_runs_06_Jul_2026.csv')
primary

,Key2,Key,Chain,Depot,PSKU,Brand,Index Rate,Portfolio,Run Month,Month Date,...,Offtake Chain depot PSKU Lag 3 Val,Offtake Chain depot PSKU P3M Val,LY Offtake Chain depot PSKU P3M Val,LY Offtake Chain depot PSKU Actuals Val,LY Offtake Chain depot PSKU Lag 1 Val,LY Offtake Chain depot PSKU Lag 2 Val,LY Offtake Chain depot PSKU Lag 3 Val,LY Offtake Chain depot PSKU Lead 1 Val,LY Offtake Chain depot PSKU Lead 2 Val,Calculated Primary Val
0,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-06-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-07-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-08-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-09-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-10-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212675,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2026-11-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
212676,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2026-12-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
212677,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2027-01-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
212678,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2027-02-28,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [72]:
primary.columns

Index(['Key2', 'Key', 'Chain', 'Depot', 'PSKU', 'Brand', 'Index Rate',
       'Portfolio', 'Run Month', 'Month Date',
       ...
       'Offtake Chain depot PSKU Lag 3 Val',
       'Offtake Chain depot PSKU P3M Val',
       'LY Offtake Chain depot PSKU P3M Val',
       'LY Offtake Chain depot PSKU Actuals Val',
       'LY Offtake Chain depot PSKU Lag 1 Val',
       'LY Offtake Chain depot PSKU Lag 2 Val',
       'LY Offtake Chain depot PSKU Lag 3 Val',
       'LY Offtake Chain depot PSKU Lead 1 Val',
       'LY Offtake Chain depot PSKU Lead 2 Val', 'Calculated Primary Val'],
      dtype='object', length=117)

In [73]:
final_df.columns = ['Chain', 'Depot', 'PSKU','Month Date','Chain_primary_vol']
final_df['Depot'] = final_df['Depot'].str.lower()
final_df

,Chain,Depot,PSKU,Month Date,Chain_primary_vol
0,Blinkit,d112,718288,2026-08-31,0.0420
1,Blinkit,d112,718312,2026-08-31,1.9460
2,Blinkit,d112,718322,2026-08-31,3.5400
3,Blinkit,d112,718328,2026-08-31,0.2151
4,Blinkit,d112,718330,2026-08-31,0.0250
...,...,...,...,...,...
1245,Zepto,d674,810518,2026-08-31,0.3080
1246,Zepto,d674,810519,2026-08-31,0.1000
1247,Zepto,d674,810673,2026-08-31,0.5040
1248,Zepto,d674,810674,2026-08-31,0.3360


In [74]:
primary['Month Date'] = pd.to_datetime(primary['Month Date'])
primary = primary.merge(final_df, on = ['Chain', 'Depot', 'PSKU','Month Date'], how = 'left')
primary['Chain_primary_val'] = primary['Chain_primary_vol']*primary['Index Rate']/10**7
primary

,Key2,Key,Chain,Depot,PSKU,Brand,Index Rate,Portfolio,Run Month,Month Date,...,LY Offtake Chain depot PSKU P3M Val,LY Offtake Chain depot PSKU Actuals Val,LY Offtake Chain depot PSKU Lag 1 Val,LY Offtake Chain depot PSKU Lag 2 Val,LY Offtake Chain depot PSKU Lag 3 Val,LY Offtake Chain depot PSKU Lead 1 Val,LY Offtake Chain depot PSKU Lead 2 Val,Calculated Primary Val,Chain_primary_vol,Chain_primary_val
0,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-06-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
1,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-07-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
2,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-08-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
3,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-09-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
4,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-10-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212675,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2026-11-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
212676,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2026-12-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
212677,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2027-01-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
212678,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2027-02-28,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN


In [75]:
primary[primary['Month Date']=='2026-08-31']['Chain_primary_val'].sum()

41.958339936520986

In [76]:
primary.to_csv('cdp_c_forecast.csv')

In [66]:
x['Chain_primary_vol'].isnull().sum()

207123

In [67]:
primary

,Key2,Key,Chain,Depot,PSKU,Brand,Index Rate,Portfolio,Run Month,Month Date,...,Offtake Chain depot PSKU Lag 3 Val,Offtake Chain depot PSKU P3M Val,LY Offtake Chain depot PSKU P3M Val,LY Offtake Chain depot PSKU Actuals Val,LY Offtake Chain depot PSKU Lag 1 Val,LY Offtake Chain depot PSKU Lag 2 Val,LY Offtake Chain depot PSKU Lag 3 Val,LY Offtake Chain depot PSKU Lead 1 Val,LY Offtake Chain depot PSKU Lead 2 Val,Calculated Primary Val
0,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-06-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-07-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-08-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-09-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-10-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212675,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2026-11-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
212676,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2026-12-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
212677,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2027-01-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
212678,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2027-02-28,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# swiggy_unpivoted.groupby(['chain_name','facility_name','item_code','date'])['forecast_quantity'].sum().reset_index()

,chain_name,facility_name,item_code,date,forecast_quantity
0,Swiggy,AHM DELHIVERY,3,2026-07-31,192
1,Swiggy,AHM DELHIVERY,3,2026-08-31,768
2,Swiggy,AHM DELHIVERY,3,2026-09-30,576
3,Swiggy,AHM DELHIVERY,102,2026-07-31,60
4,Swiggy,AHM DELHIVERY,102,2026-08-31,160
...,...,...,...,...,...
29878,Swiggy,VIZ IM1,995855,2026-08-31,0
29879,Swiggy,VIZ IM1,995855,2026-09-30,0
29880,Swiggy,VIZ IM1,999977,2026-07-31,4
29881,Swiggy,VIZ IM1,999977,2026-08-31,5


In [ ]:
blinkit_unpivoted = blinkit_unpivoted[['chain_name','facility_name','item_code','date','forecast_quantity']]
swiggy_unpivoted = swiggy_unpivoted[['chain_name','facility_name','item_code','date','forecast_quantity']]


In [ ]:
chain_forecast_unpivoted = pd.concat([blinkit_unpivoted,swiggy_unpivoted])
chain_forecast_unpivoted

,chain_name,facility_name,item_code,date,forecast_quantity
0,Blinkit,Surat S1 - Feeder Warehouse,10171388,2026-07-31,856
1,Blinkit,Surat S1 - Feeder Warehouse,10232351,2026-07-31,18
2,Blinkit,Surat S1 - Feeder Warehouse,10029462,2026-07-31,12
3,Blinkit,Surat S1 - Feeder Warehouse,10015827,2026-07-31,209
4,Blinkit,Surat S1 - Feeder Warehouse,10116052,2026-07-31,65
...,...,...,...,...,...
29878,Swiggy,PUN DELHIVERY,990631,2026-09-30,22
29879,Swiggy,CHD ECOM,991861,2026-09-30,24
29880,Swiggy,CHN ECOM,995855,2026-09-30,0
29881,Swiggy,HYD IM1,998784,2026-09-30,0


In [ ]:
len_before_merge = len(chain_forecast_unpivoted)
chain_forecast_unpivoted['item_code'] = chain_forecast_unpivoted['item_code'].astype(str)
mapping['asin'] = mapping['asin'].astype(str)
temp = mapping[['platform_name','asin','EAN','PSKU','UOM','Vol per unit']].drop_duplicates()

temp = temp[temp['platform_name'].isin(['Blinkit', 'Swiggy', 'Zepto'])]
#temp['platform_name'].unique()
duplicates = temp[temp.duplicated(subset="asin", keep=False)]
duplicates



,platform_name,asin,EAN,PSKU,UOM,Vol per unit


In [ ]:
# # Keys to match rows on
# keys = ["platform_name", "asin", "EAN", "PSKU", "UOM", "Vol per unit"]

# # Build a small DataFrame with the rows to drop
# rows_to_drop = pd.DataFrame([
#     {
#         "platform_name": "Zepto",
#         "asin": "0523a4ba-32cf-4e59-abd8-0e4086859b39",
#         "EAN": "8901088205924",
#         "PSKU": "718729",
#         "UOM": "L",
#         "Vol per unit": 100.0,
#     },
#     {
#         "platform_name": "Zepto",
#         "asin": "197827dc-3184-4c57-a966-5461967bcb3a",
#         "EAN": "8901088884402",
#         "PSKU": "808485",
#         "UOM": "L",
#         "Vol per unit": 150.0,
#     },
#     {
#         "platform_name": "Zepto",
#         "asin": "82d8e93d-3d18-44b3-9904-3bcf521d0204",
#         "EAN": "8906051370753",
#         "PSKU": "807069",
#         "UOM": "L",
#         "Vol per unit": 150.0,
#     },
#     {
#         "platform_name": "Zepto",
#         "asin": "82d8e93d-3d18-44b3-9904-3bcf521d0204",
#         "EAN": "8901088075817",
#         "PSKU": "808262",
#         "UOM": "L",
#         "Vol per unit": 150.0,
#     },
#     {
#         "platform_name": "Swiggy",
#         "asin": "944906",
#         "EAN": "8901088150095",
#         "PSKU": "718976",
#         "UOM": "L",
#         "Vol per unit": 300.0,
#     },
# ])

# # Mark rows to drop via left-merge on keys
# _marked = temp.merge(
#     rows_to_drop.assign(_drop=1),
#     on=keys,
#     how="left",
#     validate="m:m"  # remove if unsure about duplicates
# )

# # Keep everything that was not marked to drop
# temp_clean = _marked[_marked["_drop"].isna()].drop(columns=["_drop"])
# temp_clean
# duplicates = temp_clean[temp_clean.duplicated(subset="asin", keep=False)]
# duplicates

In [ ]:
temp['PSKU'] = temp['PSKU'].astype(str)
temp['EAN'] = temp['EAN'].astype(str)
temp['UOM'] = temp['UOM'].astype(str)
len_before_merge = len(chain_forecast_unpivoted)
df_chk = chain_forecast_unpivoted.merge(temp,
                  left_on = ['item_code'], right_on = ['asin'], how = 'left')
assert(len_before_merge == len(df_chk))
df_chk['date'] = pd.to_datetime(df_chk['date'])

In [ ]:
df_chk

,chain_name,facility_name,item_code,date,forecast_quantity,platform_name,asin,EAN,PSKU,UOM,Vol per unit
0,Blinkit,Surat S1 - Feeder Warehouse,10171388,2026-07-31,856,Blinkit,10171388,8901088213608,721427,TO,1000.0
1,Blinkit,Surat S1 - Feeder Warehouse,10232351,2026-07-31,18,Blinkit,10232351,8901088796804,810520,KL,1000.0
2,Blinkit,Surat S1 - Feeder Warehouse,10029462,2026-07-31,12,Blinkit,10029462,6001159111856,807033,L,125.0
3,Blinkit,Surat S1 - Feeder Warehouse,10015827,2026-07-31,209,Blinkit,10015827,8901088043953,718312,KL,1000.0
4,Blinkit,Surat S1 - Feeder Warehouse,10116052,2026-07-31,65,Blinkit,10116052,8901088205993,721133,L,300.0
...,...,...,...,...,...,...,...,...,...,...,...
50086,Swiggy,PUN DELHIVERY,990631,2026-09-30,22,NaN,NaN,NaN,NaN,NaN,NaN
50087,Swiggy,CHD ECOM,991861,2026-09-30,24,Swiggy,991861,8906027074531,729893,L,240.0
50088,Swiggy,CHN ECOM,995855,2026-09-30,0,NaN,NaN,NaN,NaN,NaN,NaN
50089,Swiggy,HYD IM1,998784,2026-09-30,0,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
duplicates = df_chk[df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False)]
duplicates.isnull().sum()

chain_name               0
facility_name            0
item_code                0
date                     0
forecast_quantity        0
platform_name        12111
asin                 12111
EAN                  12111
PSKU                 12111
UOM                  12111
Vol per unit         12111
dtype: int64

In [ ]:
df_chk = df_chk.dropna(subset = ['PSKU'])
df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False).sum()

474

In [ ]:
# duplicates = df_chk[df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False)]
# duplicates.sort_values(by=['chain_name','facility_name','PSKU','date'])[:60]

,chain_name,facility_name,item_code,date,forecast_quantity,platform_name,asin,EAN,PSKU,UOM,Vol per unit
20560,Swiggy,AHM DELHIVERY,554793,2026-07-31,0,Swiggy,554793,8901088886970,809042,KG,225.0
23718,Swiggy,AHM DELHIVERY,60103,2026-07-31,60,Swiggy,60103,8901088886970,809042,KG,225.0
30521,Swiggy,AHM DELHIVERY,554793,2026-08-31,0,Swiggy,554793,8901088886970,809042,KG,225.0
33679,Swiggy,AHM DELHIVERY,60103,2026-08-31,60,Swiggy,60103,8901088886970,809042,KG,225.0
40482,Swiggy,AHM DELHIVERY,554793,2026-09-30,0,Swiggy,554793,8901088886970,809042,KG,225.0
43640,Swiggy,AHM DELHIVERY,60103,2026-09-30,120,Swiggy,60103,8901088886970,809042,KG,225.0
21827,Swiggy,BLR DHL,819548,2026-07-31,192,Swiggy,819548,8901088171755,719162,TO,250.0
23859,Swiggy,BLR DHL,298412,2026-07-31,0,Swiggy,298412,8901088171755,719162,TO,250.0
31788,Swiggy,BLR DHL,819548,2026-08-31,192,Swiggy,819548,8901088171755,719162,TO,250.0
33820,Swiggy,BLR DHL,298412,2026-08-31,0,Swiggy,298412,8901088171755,719162,TO,250.0


In [ ]:
# duplicates.sort_values(by=['chain_name','facility_name','PSKU','date']).to_csv('duplicates_swiggy2.csv')

In [ ]:
# df_chk = df_chk.sort_values('forecast_quantity', ascending=False) \
#        .drop_duplicates(subset=['chain_name', 'facility_name','PSKU' , 'date'], keep='first')
# df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False).sum()

0

In [ ]:
df_chk.columns

Index(['chain_name', 'facility_name', 'item_code', 'date', 'forecast_quantity',
       'platform_name', 'asin', 'EAN', 'PSKU', 'UOM', 'Vol per unit'],
      dtype='object')

In [ ]:
df_chk['month_date'] = df_chk['date'] + pd.offsets.MonthEnd(0)

df_chk.rename(columns = {'item_code':'platform_code', 'EAN':'eancode', 'UOM':'uom_reporting',
                         'Vol per unit':'vol_per_unit'},inplace=True)
df_chk['vol_in_lit'] = df_chk['forecast_quantity']*df_chk['vol_per_unit']/1000
df_chk['vol_in_rum'] = df_chk.apply(
    lambda x: x['vol_in_lit'] / 1000 if x['uom_reporting'] in ['KL', 'TO'] else x['vol_in_lit'],
    axis=1
)

df_chk = df_chk.groupby(['chain_name','facility_name', 'PSKU','month_date'])[['vol_in_rum']].sum().reset_index()
df_chk['PSKU'] = df_chk['PSKU'].astype(int)
df_chk.rename(columns = {'PSKU':'parent_material_code'}, inplace = True)
df_chk

,chain_name,facility_name,parent_material_code,month_date,vol_in_rum,forecast_quantity
0,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-07-31,5.754,959
1,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-08-31,6.204,1034
2,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-09-30,6.222,1037
3,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-10-31,8.568,1428
4,Blinkit,Ahmedabad A2 - Feeder Warehouse,718312,2026-07-31,0.501,501
...,...,...,...,...,...,...
37674,Swiggy,VIZ IM1,810685,2026-08-31,0.008,20
37675,Swiggy,VIZ IM1,810685,2026-09-30,0.008,20
37676,Swiggy,VIZ IM1,810738,2026-07-31,0.000,0
37677,Swiggy,VIZ IM1,810738,2026-08-31,0.000,0


In [ ]:
df_chk.duplicated(subset=['chain_name','facility_name','parent_material_code','month_date'], keep=False).sum()

0